# Prompting Pipeline: Extracting Variables from NSW Coronial Findings

This notebook sends coronial findings reports (PDFs) to an OpenAI model and asks it to extract 28 variables from each report (for example age at death, cause of death, medications and eight flags). Three **prompting techniques** are compared:

| Technique | What the model is given |
|---|---|
| **Zero-shot** | Instructions and the report. No examples. |
| **Few-shot** | The same instructions, one or two *worked examples* (a report together with its correct answer), then the report to extract. |
| **Chain-of-thought (CoT)** | The same instructions and a request to reason before giving the answer. |

Each API call is a **trial**: one repeat of one report with one prompt. Every trial is saved to an **audit log** that records the exact prompt text, the model's answer and the number of tokens used. The audit log is the main output of this notebook and the input of the evaluation notebook (`evaluation_pipeline_final.ipynb`).

Report links are read from `report_links.jsonl` (one line per report). A report marked `"status": "withdrawn"` in that file is skipped (Section 3). The links are checked separately in `check_report_links_final.ipynb`.

## Structure of the notebook

| Section | Content |
|---|---|
| 1 to 4 | Setup, the 28-variable schema, the reports, and the components from which prompts are built. |
| 5 | How a prompt reaches the model (system, user and assistant messages). |
| 6 | What is saved for each trial, and how **changes to a prompt are recorded automatically** (prompt versions). |
| 7 | Three helper functions shared by all techniques: the dataset check, the API call and the logging. |
| 8 | **Zero-shot:** the messages, the trial function, the prompts, the smoke test and the run. |
| 9 | **Few-shot:** the same, together with the worked examples. |
| 10 | **Chain-of-thought:** the same. |
| 11 and 12 | Export of the results, and the steps after prompt development. |

## The three datasets

The 50 reports are divided into three datasets, and the notebook runs only the datasets that are currently allowed (Section 3).

| Dataset | Reports | Repeats per prompt and report | Purpose |
|---|---|---|---|
| **Prompt development** | 5 | 5 | All prompts are run and compared. |
| **Internal validation** | 5 | 5 | The best prompt of each technique is run, and the single best prompt is selected. |
| **External validation** | 40 | 1 | Held out until the end. Only the selected prompt is run, once per report. |

## Test mode

`TEST_MODE = True` runs the pipeline on one test report that is not one of the 50 study reports, so that the pipeline can be checked without using the study reports. Test results are saved in a separate folder, `results_test/`. For the study runs, `MODEL` is set to the study model and `TEST_MODE = False`.

## Requirements

Python 3.11.15 with `openai` 2.45.0, `pydantic` 2.13.4, `python-dotenv` 1.2.2, `pandas` 3.0.3 and `openpyxl` 3.1.5 (the versions used in the study). An OpenAI API key is required in a `.env` file (`OPENAI_API_KEY`). The notebook is run from the `code/` folder of the repository: it reads the report links from `data/report_links.jsonl` and writes its outputs to `results/` (or `results_test/` in test mode).

## 1. Setup

Imports, the API key from the `.env` file, and the two settings that control a run: `MODEL` and `TEST_MODE`. The request settings are fixed for the whole study: `REASONING_EFFORT = "medium"` (the documented default of the model) and `MAX_OUTPUT_TOKENS = 30000`. `TEMPERATURE`, `TOP_P` and `VERBOSITY` are not sent, so the model's defaults apply; the API does not accept temperature or top_p when the reasoning effort is not "none". The settings are identical for every trial, so that the prompt is the only element that differs, and they are saved with each trial. Section 1.1 checks which settings the model accepts.

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel
from pathlib import Path
from typing import Literal, get_args
from datetime import datetime
import hashlib
import json
import os
import time
import pandas as pd

print(f"Loading .env from: {find_dotenv()}")
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in your environment (for example in a .env file) before running this notebook.")

# THE MODEL
# One model is used for every study run; a smaller model was used only to test the pipeline.
# Testing: a smaller model and TEST_MODE = True. Study runs: the study model and TEST_MODE = False.
# MODEL = "gpt-5.4-mini-2026-03-17"
MODEL = "gpt-5.6-luna"
TEST_MODE = False

# REQUEST SETTINGS (identical for every trial)
# Held constant across all prompts, so that the prompt is the only element that differs. None means "not sent" (the model's default applies). The settings are saved with every trial (PromptTrialLog), together with the settings the API reports it applied.
REASONING_EFFORT = "medium"    # fixed for the whole study: the documented default of gpt-5.6-luna (levels: none, low, medium, high, xhigh, max)
TEMPERATURE = None             # not sent: the API does not accept temperature or top_p when the reasoning effort is not "none" (see check_parameters() in Section 1.1)
TOP_P = None
MAX_OUTPUT_TOKENS = 30000      # upper limit, identical for every trial; hidden reasoning tokens count towards it (test runs used fewer than 3,000)
VERBOSITY = None               # "low" / "medium" / "high"; None = the model's default

# The Responses API has no `seed` parameter and its response has no `system_fingerprint` field (both belong to the Chat Completions API). The model name that served each call is saved instead (`model` in PromptTrialLog).

if not TEST_MODE and "mini" in MODEL:
    raise RuntimeError("TEST_MODE is False but MODEL is a mini model: set MODEL to the chosen full model for the real experiment.")

if not TEST_MODE and REASONING_EFFORT is None:
    raise RuntimeError("TEST_MODE is False but REASONING_EFFORT is None: set one fixed level in Section 1 so that it is identical for every trial.")

LINKS_FILE = Path("../data/report_links.jsonl")                       # the report links, one line per report
RESULTS_ROOT = Path("../results_test") if TEST_MODE else Path("../results")

client = OpenAI(api_key=OPENAI_API_KEY, timeout=600, max_retries=3)   # the SDK retries temporary errors automatically
print(f"MODEL = {MODEL} | TEST_MODE = {TEST_MODE} | results are saved in: {RESULTS_ROOT}/")

### 1.1 Parameter check

`check_parameters()` sends a few short, text-only requests (no reports and no study data) to the model, each with one request setting added, and prints which settings the API accepts or rejects. The check is needed because the sampling settings planned at the start of the study (`temperature`, `top_p`) are rejected by many reasoning models, and the Responses API has no `seed` parameter. The result is appended to `parameter_check.csv` in the results folder, so that the settings available for the model are documented. The check makes about seven short calls.

In [ ]:
def check_parameters() -> pd.DataFrame:
    """Test which request settings the chosen model accepts, using very short text-only requests.

    One request is sent without any setting (baseline) and one for each setting listed below. A setting that the API rejects is reported with the error message. Nothing is saved with the trials; the result is printed and appended to `parameter_check.csv` in the results folder.

    Returns:
        One row per tested setting: whether it was accepted, and what the API reported (or the error message).
    """
    candidates = {
        "no setting (baseline)": {},
        "reasoning effort = low": {"reasoning": {"effort": "low"}},
        "temperature = 0": {"temperature": 0},
        "top_p = 1": {"top_p": 1},
        "verbosity = low": {"text": {"verbosity": "low"}},
        "max_output_tokens = 2000": {"max_output_tokens": 2000},
        "seed = 1": {"extra_body": {"seed": 1}},
    }
    rows = []
    for name, kwargs in candidates.items():
        try:
            r = client.responses.create(model=MODEL, input="Reply with the single word: ready.", **kwargs)
            reasoning = getattr(r, "reasoning", None)
            detail = (f"model={r.model}; status={getattr(r, 'status', None)}; temperature={getattr(r, 'temperature', None)}; top_p={getattr(r, 'top_p', None)}; reasoning effort={getattr(reasoning, 'effort', None)}")
            rows.append({"setting": name, "accepted": True, "detail": detail})
        except Exception as e:
            message = str(getattr(e, "message", e)).replace("\n", " ")
            rows.append({"setting": name, "accepted": False, "detail": message[:200]})
    result = pd.DataFrame(rows)
    print(result.to_string(index=False))
    saved = result.assign(model=MODEL, checked_at=str(datetime.now()))
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    path = RESULTS_ROOT / "parameter_check.csv"
    saved.to_csv(path, mode="a", header=not path.exists(), index=False)
    return result

In [ ]:
# check_parameters();

## 2. Schema: the 28 variables

`VariablesToExtractExtended` lists the 28 variables. It is passed to the API as the required output format (`text_format`, Section 5), so every answer is returned as valid, correctly typed JSON, whichever prompt produced it.

- Flags are `bool | None`: `True` or `False` only when the report states it explicitly, and `None` (blank) when the report does not address the topic (the explicit-only rule).
- Variables with several values (for example medications and conditions) are lists of text; the other variables are text in the report's own wording.

Two further schemas are used only for chain-of-thought (Section 10). `ExtractionWithReasoning` adds a `reasoning` text field before the same 28 variables. `ExtractionWithEvidence` adds a structured list of evidence instead (field, quotation and rule applied), and is used by COT1_struct and COT2_C.

In [ ]:
class VariablesToExtractExtended(BaseModel):
    file_number: str                       # one file number per report
    sex: str
    age_at_death: str                      # text: may be an estimate or a range, for example "47 or 48"
    date_of_findings: str                  # text, in the report's own wording
    date_of_death: str                     # text, in the report's own wording (for example "April 2013", a range, or blank)
    place_of_death: str
    definitive_cause_of_death: str
    potential_causes_of_death: list[str]
    manner_of_death: str                   # the coroner's wording with the name and pronouns removed; blank if none
    country_of_birth: str
    place_of_residence: str                # most recent residence as stated; a region is acceptable
    profession: str                        # most recent occupation, if stated; otherwise blank
    level_of_education: str
    medications: list[str]
    intellectual_disability_flag: bool | None
    intellectual_disability_conditions: list[str]   # verbatim
    mental_health_flag: bool | None
    mental_health_conditions: list[str]             # verbatim
    substance_use_flag: bool | None        # abuse, misuse or dependence on alcohol or drugs, or smoking (see the definitions in Section 4)
    substances_used: list[str]
    known_health_issues: list[str]
    custody_served_flag: bool | None
    non_english_speaker_flag: bool | None
    use_of_interpreters_flag: bool | None
    preventable_death_flag: bool | None
    domestic_violence_flag: bool | None    # reliability discussed in the dissertation (Section 4.5.3)
    toxicology_results_summary: str        # verbatim (name and pronouns removed); blank if none
    coroners_recommendations_summary: str  # verbatim, including the recipient heading (name and pronouns removed); blank if none


class ExtractionWithReasoning(BaseModel):
    """Chain-of-thought output shape: the model's reasoning, followed by the same 28-variable extraction."""
    reasoning: str
    extraction: VariablesToExtractExtended

# 28 variable names, written out so that the type can be checked; the assert below stops the notebook if the list and the schema differ.
VariableName = Literal[
    "file_number",
    "sex",
    "age_at_death",
    "date_of_findings",
    "date_of_death",
    "place_of_death",
    "definitive_cause_of_death",
    "potential_causes_of_death",
    "manner_of_death",
    "country_of_birth",
    "place_of_residence",
    "profession",
    "level_of_education",
    "medications",
    "intellectual_disability_flag",
    "intellectual_disability_conditions",
    "mental_health_flag",
    "mental_health_conditions",
    "substance_use_flag",
    "substances_used",
    "known_health_issues",
    "custody_served_flag",
    "non_english_speaker_flag",
    "use_of_interpreters_flag",
    "preventable_death_flag",
    "domestic_violence_flag",
    "toxicology_results_summary",
    "coroners_recommendations_summary",
]
assert set(get_args(VariableName)) == set(VariablesToExtractExtended.model_fields), "VariableName is out of sync with the schema"

class EvidenceItem(BaseModel):
    """One line of evidence: the field, the words of the report that bear on it, and the rule or card item that applies."""
    field: VariableName                 # only the 28 variable names are accepted
    quote: str
    rule_applied: str


class ExtractionWithEvidence(BaseModel):
    """Structured chain-of-thought output shape: a list of evidence items, followed by the same 28-variable extraction."""
    evidence: list[EvidenceItem]
    extraction: VariablesToExtractExtended

## 3. Reports and the datasets allowed to run

The report links are read from `report_links.jsonl`: one line per report, each a JSON object with the report number, its dataset (`prompt_development`, `internal_validation` or `external_validation`), its file number and its link. The file number allows every trial to check that the link opened the intended report.

A report withdrawn by the court keeps its line, with the additional entries `"status": "withdrawn"` and `"note"`. A withdrawn report is never run (it is excluded from its dataset and a trial on it is refused), but its file number is still used in the check that no worked example is a study report. Lines of other sets (the worked examples and the test report, used by `check_report_links_final.ipynb`) are ignored here.

**`ALLOWED_DATASETS` is the only switch.** A trial on a report outside the allowed datasets stops with an error before anything is sent to the API. The switch was moved on in this order:

1. Prompt development only.
2. After the best prompt of each technique was chosen, `"internal_validation"` was added and those three prompts were run.
3. After the single best prompt was chosen, `"external_validation"` was added and that prompt was run once per report. The external validation reports were not used before this point.

Because every trial stores the full text of its prompt, the prompt run on the external validation reports can be checked against the prompt chosen at internal validation.

In [ ]:
STUDY_SETS = ["prompt_development", "internal_validation", "external_validation"]


def load_reports() -> dict[int, dict]:
    """Read the report list from the links file.

    Lines whose set is not one of the three study sets (the worked examples and the test report, which are only used by the link checker) are ignored.

    Returns:
        A dict mapping each report number to a dict with the keys `dataset`, `file_number`, `file_url` (None when the line gives no link) and `status` ("active", or "withdrawn" for a report the court has removed).

    Raises:
        ValueError: If a report number appears on more than one line.
    """
    reports = {}
    for line in open(LINKS_FILE, encoding="utf-8"):
        if not line.strip():
            continue
        entry = json.loads(line)
        if entry["dataset"] not in STUDY_SETS:
            continue
        report_no = int(entry["report_no"])
        if report_no in reports:
            raise ValueError(f"Report {report_no} appears more than once in {LINKS_FILE}.")
        reports[report_no] = dict(dataset=entry["dataset"].strip(), file_number=entry["file_number"].strip(),
                                  file_url=(entry.get("file_url") or "").strip() or None, status=entry.get("status", "active"))
    return reports

REPORTS = load_reports()
print(f"{len(REPORTS)} study reports read from {LINKS_FILE}" + ''.join(f" | report {n} is withdrawn and will not be run" for n, r in REPORTS.items() if r['status'] == 'withdrawn'))

# Test report, used only to test the pipeline; not one of the 50 study reports (file 2017/00335331).
TEST_REPORT_NO = 900
REPORTS[TEST_REPORT_NO] = dict(dataset="test_report", file_number="2017/00335331", status="active",
    file_url="https://coroners.nsw.gov.au/documents/findings/2022/Inquest_into_the_death_of_Riley_Christopher_Shortland.pdf")

**Datasets and repeats.** `ALLOWED_DATASETS` lists the datasets that may run. `REPEATS_PER_DATASET` gives the number of repeats of each prompt on each report: 5 for prompt development and internal validation, and 1 for external validation. `DATASET` is the dataset that is run; after it is changed, running the cell again updates `TARGET_REPORTS` and `N_REPEATS`.

In [ ]:
# Datasets that may run
ALLOWED_DATASETS = {"test_report"} if TEST_MODE else {"test_report", "internal_validation", "external_validation"}   # prompt development and internal validation are closed; external validation runs the selected prompt (Z7_v2)

# Repeats of each prompt on each report, by dataset
REPEATS_PER_DATASET = {"test_report": 5, "prompt_development": 5, "internal_validation": 5, "external_validation": 1}

# The dataset to run (after changing DATASET, run this cell again)
DATASET = "test_report" if TEST_MODE else "external_validation"


def reports_in(dataset: str) -> list[int]:
    """Return the report numbers that belong to one set.

    Args:
        dataset: Name of the set, for example "prompt_development".

    Returns:
        The report numbers of that set, in ascending order, without any report that has been withdrawn.
    """
    return sorted(n for n, r in REPORTS.items() if r["dataset"] == dataset and r["status"] != "withdrawn")


TARGET_REPORTS = reports_in(DATASET)
N_REPEATS = REPEATS_PER_DATASET[DATASET]
print(f"{DATASET}: reports {TARGET_REPORTS}, {N_REPEATS} repeat(s) of each prompt on each report")
for _d in ["prompt_development", "internal_validation", "external_validation"]:
    _missing = [n for n in reports_in(_d) if not REPORTS[n]["file_url"]]
    print(f"  {_d}: {len(reports_in(_d))} reports" + (f", links missing for {_missing}" if _missing else ", all links present"))

## 4. Prompt components

A prompt is built from the components below. Each entry of `COMPONENTS` is a separate block of instruction text that can be switched on or off, which allows prompts to be compared. `TASK_CORE` is the short task statement (what to extract and how to treat missing information). `COT_INSTRUCTION` is added only for chain-of-thought.

In [ ]:
DECEDENT_RULE = ("For manner_of_death and toxicology_results_summary, use the coroner's own wording but remove the deceased's name and any title, pronouns or other references to the deceased, so that the answer reads as a subjectless phrase. This rule does not apply to coroners_recommendations_summary, which is copied exactly as written, including any name of the deceased. All other fields are recorded as stated in the report.")

BOOLEAN_RULE = ("For every boolean (*_flag) field below: return TRUE only if the report explicitly and directly confirms the fact, and FALSE only if the report explicitly and directly denies or negates it. If the report does not address the topic at all, leave the field blank. Do not return FALSE by default, and do not infer a value from indirect or contextual information, or from sources outside of the provided report.")

EMPTY_RULE = ("If the report does not provide the information for a field, leave the field empty: return null for a boolean flag, an empty string for a text field, and an empty list for a list field. Do not write placeholders such as 'N/A', 'unknown' or 'not stated'.")

LIST_RULE = ("For every list field, give each item as a short phrase, using the report's own terminology where possible. A comma inside an item is part of that item and does not start a new item.")

FIELD_DEFINITIONS = {
    "file_number": "The unique case identifier assigned by the coroner's court. The format is typically a year followed by a case number, with a forward slash separator.",
    "sex": "The deceased's sex, only if explicitly stated or consistently indicated through pronoun use. Do not infer from a name.",
    "age_at_death": "The deceased's age in whole years at time of death, from an explicit statement or calculable from an explicitly stated date of birth and date of death. If only an estimated age or age range is given, record the estimate exactly as stated in the report.",
    "date_of_findings": "The date the coroner delivered or published the findings, as given by the date of findings field in the report. Record the date only, in the report's own format and wording (for example '3 March 2020'); do not reformat it. Ignore any later corrigendum date.",
    "date_of_death": "The date, estimated date, or date range on which the deceased died, as stated by the coroner. Record the date together with any qualifying wording exactly as written in the report (for example 'on or about 12 June 2015' or 'between March 2016 and 9 May 2016'). Leave out the verb ('died'), the deceased's name and any other surrounding words. Do not resolve a qualified date to a single date and do not reformat it. If the report gives no date, leave it empty.",
    "place_of_death": "The location where the deceased died, or was found if the exact location of death is unknown. Record the location as specifically as the report states it, in the report's own wording.",
    "definitive_cause_of_death": "The specific, medically or forensically ascertained cause of death, stated by the coroner as an established fact. Leave blank if the coroner does not reach a definite conclusion. Use potential_causes_of_death instead if no definitive cause of death is given and potential causes are given in the report directly. Never populate both fields for the same report.",
    "potential_causes_of_death": "A list of possible, unconfirmed causes of death stated by the coroner, used only when no definitive cause is reached. The potential causes should be directly referenced within the report; do not make up or infer potential causes. Do not populate definitive_cause_of_death if no definitive cause is given and only potential causes are stated in the report.",
    "manner_of_death": "The coroner's stated manner of death, in the coroner's own wording, with the decedent's name and pronouns removed so that it reads as a subjectless phrase (see the general rule on decedent references). If no manner of death is given in the report, do not infer one from the rest of the report; leave it empty.",
    "country_of_birth": "The country the deceased was born in, if explicitly stated. Do not infer from surname, cultural references, or place of residence.",
    "place_of_residence": "The deceased's most recent place of residence before death, exactly as stated in the report. A region or area is acceptable if the report gives nothing more specific.",
    "profession": "The deceased's most recent occupation or status (e.g. a stated profession, or \"retired\"/\"unemployed\"/\"student\" if that was their status at time of death).",
    "level_of_education": "The deceased's highest confirmed educational achievement. If a qualification was started but not completed, state this explicitly rather than recording it as achieved.",
    "medications": "A list of every medication named in the report that the deceased was taking or was prescribed, at any time. Record names only, without doses, amounts or frequencies. List every name the report uses, brand and generic alike, even when two names refer to the same drug. The name of a class of medication (for example 'statins' or 'antibiotics') is acceptable, including when it is the only medication information the report gives. Do not include a substance only because it was detected in toxicology screening, unless the report also describes it as a prescribed or taken medication.",
    "intellectual_disability_flag": "Whether the report explicitly documents a diagnosed or clearly described intellectual, developmental, or learning disability.",
    "intellectual_disability_conditions": "If intellectual_disability_flag is true, the specific condition(s) named, using the report's own terminology.",
    "mental_health_flag": "Whether the report explicitly documents a diagnosed or clearly described mental health condition.",
    "mental_health_conditions": "If mental_health_flag is true, the specific condition(s) named, using the report's own terminology.",
    "substance_use_flag": "Whether the report explicitly states that the deceased abused, misused or was dependent on alcohol or any other drug (for example a diagnosed substance use disorder, addiction, a drug or alcohol problem, or heavy, regular or problematic use), or smoked tobacco or used nicotine products. Social, occasional or experimental use of alcohol or other drugs does not count. A medication taken as prescribed does not count, and a substance mentioned only in toxicology results, without a statement about the deceased's use of it, does not count on its own. False only if the report explicitly states that the deceased did not misuse such substances.",
    "substances_used": "If substance_use_flag is true, the alcohol, tobacco and other drugs that the report names as abused, misused or relied on by the deceased (and tobacco or nicotine where the deceased smoked or used it), using the report's own terminology. If the substance is not named, give the general term the report uses followed by '(unspecified)'. Leave empty otherwise.",
    "known_health_issues": "A list of the deceased's known health conditions, including physical or chronic conditions, injuries, symptoms and acute clinical findings, and including any conditions listed under intellectual_disability_conditions or mental_health_conditions. Each item should be a short phrase. Conditions that result from substance use (for example alcohol withdrawal) are health issues and are included. Do not include substance use or dependence itself, or lifestyle behaviours such as smoking or drinking.",
    "custody_served_flag": "Whether the report explicitly states the deceased served a custodial sentence or was held in custody at any point, including police or correctional custody at the time of death. Being charged, arrested, or having outstanding warrants alone does not qualify.",
    "non_english_speaker_flag": "Whether the report explicitly states the deceased was not a native or fluent English speaker, or had English as an additional language. Do not infer this from country of birth or name alone.",
    "use_of_interpreters_flag": "Whether the report explicitly states an interpreter was used at any point connected to the case.",
    "preventable_death_flag": "Whether the coroner explicitly and directly states the death was preventable. The coroner making recommendations alone does not satisfy this. An explicit statement on preventability is required.",
    "domestic_violence_flag": "Whether the report describes a domestic violence related charge or conviction involving the deceased, an apprehended domestic violence order involving the deceased, or a direct statement that the deceased experienced or perpetrated domestic violence.",
    "toxicology_results_summary": "The toxicology findings copied from the report in the coroner's own wording, including the substances detected and the levels at which they were found, with the deceased's name, title and any pronouns removed (see the general rule on decedent references). If no toxicology findings are included in the report, leave empty.",
    "coroners_recommendations_summary": "The coroner's formal recommendations copied verbatim from the report, covering every distinct recommendation and including any name of the deceased exactly as written. Include the heading naming the recipient(s) of the recommendations (for example 'To the Chief Executive, ...') and keep the original numbering. Do not include general observations or notes that are not formal recommendations. If the report makes no recommendations, leave empty.",
}

# Check: every schema field has a definition and every definition has a field, so that the schema and the definitions cannot differ.
assert list(FIELD_DEFINITIONS) == list(VariablesToExtractExtended.model_fields), \
    "FIELD_DEFINITIONS and the schema are out of sync"

def _build_variable_definitions() -> str:
    """Join the general rules and the field definitions into one block of text.

    Returns:
        DECEDENT_RULE, BOOLEAN_RULE, EMPTY_RULE and LIST_RULE, followed by one "field: definition" entry per field of FIELD_DEFINITIONS, separated by spaces.
    """
    parts = [DECEDENT_RULE, BOOLEAN_RULE, EMPTY_RULE, LIST_RULE]
    parts += [f"{name}: {text}" for name, text in FIELD_DEFINITIONS.items()]
    return " ".join(parts)

COMPONENTS = {
    "role_definition": ("You are an expert at structured data extraction. You will be given unstructured text from a coronial findings and recommendations report and must convert it into the given structure."),

    "mhc_id_icd_definitions": ("According to the ICD-11, a mental health condition is a clinically significant disturbance in cognition, emotional regulation, or behaviour that reflects dysfunction in psychological, biological, or developmental processes and causes distress or impairment. An intellectual disability is defined as a developmental condition characterised by significant limitations in intellectual functioning (reasoning, problem-solving, learning) and adaptive behaviour, originating during the developmental period."),

    "date_format_instruction": ("Report all dates exactly as they appear in the source document; do not reformat or infer missing date components."),

    "acronym_instruction": ("For diagnostic labels, use the full clinical term rather than an acronym (e.g., 'Attention-Deficit/Hyperactivity Disorder', not 'ADHD'), unless the source document uses only the acronym and gives no full form."),

    "variable_definitions": _build_variable_definitions(),

    # toxicology and recommendations are extracted verbatim, so this instruction covers the list fields only
    "detail_level_instruction": ("For list-type fields (conditions, medications, known health issues, potential causes of death), keep each item to a short phrase found in the report rather than a full sentence. For all other fields, extract only what is explicitly stated, without inferred elaboration."),
}

In [ ]:
# The variable list is generated from the schema, so that the task text always matches it.
_FIELD_LIST = ", ".join(VariablesToExtractExtended.model_fields)

TASK_CORE = (f"Extract the following medical and demographic information from the given coronial findings and recommendations report. Only extract information present in the report. If a field is not present or unknown, return an empty string (or an empty list for list fields), and null for boolean flags. Outputs must be accurate and concise; do not provide additional commentary or explanation. Do not provide potential speculative information when definitive information is in the report. Do not provide additional information beyond what is requested. Extract the following fields: {_FIELD_LIST}.")


COT_INSTRUCTION = ("Before producing your final answer, reason step by step through the report to determine each field's value. Put that reasoning in the `reasoning` field, then put the completed extraction in the `extraction` field.")

### 4.1 Decision cards

A **decision card** restates one variable as its definition, what counts, what does not count, how to treat a borderline case and what to record when the report is silent (dissertation, Section 2.3.1.1 and Appendix G.4). Two components use the cards:

- `decision_cards` (prompt **Z5**) adds short cards for ten variables to the Z4 instructions. These cards have no definition line, so they keep the "Counts" line.
- `variable_cards` (prompts **Z6** and **Z7**) gives a card for each of the 28 variables in place of the Z4 definition list, together with the general rules (`general_rules`). The definition line of each card is copied from `FIELD_DEFINITIONS`, so a card and the Z4 definition always have the same wording. To avoid repetition, a full card keeps the "Counts" line only where it adds to the definition (`COUNTS_ADDS`: sex, country of birth and the intellectual disability flag).

The texts of the cards are in `CARD_RULES`. The cards for the mental health and intellectual disability flags follow the boundaries in the data dictionary (dissertation, Appendix C.2). Where an annotation rule is not part of the Z4 definition, it is stated in the card; for example, a place of birth in Australia gives the country "Australia", because reports from New South Wales usually give only the place.

In [ ]:
# DECISION CARDS
# A card states, for one variable, what counts, what does not count, how to treat a borderline case and what to record when the report is silent. The definition line of a card is copied from FIELD_DEFINITIONS, so a card and the Z4 definition always have the same wording. The four texts of each card are in CARD_RULES.
# Order of the texts: counts, does not count, borderline, if the report is silent.
# In the full cards (Z6 and Z7) the "Counts" line is left out when it only repeats the definition (see COUNTS_ADDS below).
CARD_RULES = {
    "file_number": ("the court file number printed in the report, in the form year/number (for example 2021/123456).", "any other reference number.", "copy the number exactly as printed, including any leading zeros.", "leave empty."),
    "sex": ("an explicit statement of sex, or consistent pronouns and titles (\"he\", \"his\", \"Mr\" for Male; \"she\", \"her\", \"Mrs\", \"Ms\" for Female).", "an inference from the name alone.", "if a statement and the pronouns differ, use the explicit statement.", "leave empty."),
    "age_at_death": ("the age in whole years stated in the report, or calculated from an explicitly stated date of birth and date of death.", "an age at another event, such as at diagnosis or at an earlier incident.", "an estimate or range is copied as stated (for example \"47 or 48\"). Give digits only (for example \"43\"), not \"43 years old\" or \"forty-three\".", "leave empty."),
    "date_of_findings": ("the date the coroner delivered or published the findings, as given in the date of findings of the report.", "the hearing date, the date of death, a later corrigendum date, or any other date in the report.", "record the date only. If only a month and year are given, copy them as written. Do not reformat the date, and ignore any later corrigendum date.", "leave empty."),
    "date_of_death": ("the date, estimated date or date range on which the deceased died, as stated by the coroner, with any qualifying wording.", "the date of findings, or the date of an earlier incident or admission that is not the date of death.", "keep qualifying wording such as \"on or about\" or \"between ... and ...\" exactly as written; do not resolve it to a single date. Leave out the verb (\"died\"), the deceased's name and any other surrounding words. Do not reformat the date.", "leave empty."),
    "place_of_death": ("the location where the deceased died, or was found if the exact place of death is unknown, as specifically as the report states it.", "the place of residence, the place of an earlier incident, or the place of the hearing.", "use the report's own wording, including a town or postcode if given.", "leave empty."),
    "definitive_cause_of_death": ("a cause of death that the coroner states as an established fact.", "a cause the coroner describes as possible, probable or unconfirmed, and the manner of death.", "if the coroner reaches no definite conclusion, leave this field empty and use potential_causes_of_death. Never fill both fields for the same report.", "leave empty."),
    "potential_causes_of_death": ("possible, unconfirmed causes of death that the report directly states, used only when no definitive cause is reached.", "any cause when a definitive cause is given, and any cause that is inferred and not stated.", "when a definitive cause of death is given, return an empty list.", "return an empty list."),
    "manner_of_death": ("the manner of death as stated by the coroner, in the coroner's own wording.", "the cause of death, and a manner inferred from other parts of the report.", "remove the deceased's name, title and pronouns so that the answer is a subjectless phrase. If the report states no manner of death, leave empty even if one could be inferred.", "leave empty."),
    "country_of_birth": ("the country of birth as named in the report (for example \"China\" or \"India\"). Exception: a town, suburb or other place of birth in Australia gives \"Australia\" even though the country is not named (for example \"Woy Woy\" gives \"Australia\"), because reports from New South Wales usually give only the place.", "an inference from the surname, cultural background or place of residence.", "record the country only, not the city or region (for example \"Chengdu, China\" gives \"China\"). A place of birth outside Australia with no country named is recorded as the place name written in the report.", "leave empty."),
    "place_of_residence": ("the most recent place of residence before death, as stated in the report.", "a custody facility, a hospital or the place of death, and an earlier residence when a later one is given.", "if only an area or region is given, record that area.", "leave empty."),
    "profession": ("the most recent occupation or status of the deceased at the time of death (for example \"retired\", \"unemployed\", \"student\").", "an earlier job when the report gives a later occupation or status.", "work described only as history (for example \"worked as a plumber until 2015\") is recorded only if the report says it was the occupation at the time of death; otherwise leave empty.", "leave empty."),
    "level_of_education": ("the highest educational achievement that the report confirms.", "a qualification that was started and not completed, when recorded as if achieved.", "if a course was started but not completed, say so explicitly (for example \"commenced university studies but ceased shortly afterwards\").", "leave empty."),
    "medications": ("every medication named in the report that the deceased was taking or was prescribed, at any time: brand and generic names alike, even when two names refer to the same drug, and the name of a class of medication (for example statins or antibiotics). Names only, without doses, amounts or frequencies.", "a substance detected only in toxicology screening, unless the report also describes it as a prescribed or taken medication.", "a general mention such as \"prescribed medication\" with no medication or class named gives no item. Do not record doses, amounts or frequencies.", "return an empty list."),
    "intellectual_disability_flag": ("a stated diagnosis of an intellectual, developmental or learning disability, including dementia.", "cognitive difficulties, and a suspected or possible neurocognitive disorder.", "a diagnosis described only as possible or queried does not count; list it under known_health_issues instead.", "null. False only if the report explicitly denies it."),
    "intellectual_disability_conditions": ("the specific condition(s) named, using the report's own terminology, when intellectual_disability_flag is true.", "cognitive difficulties or conditions the report does not name as a diagnosis.", "use the report's wording; do not substitute a more general or a different term.", "return an empty list."),
    "mental_health_flag": ("a diagnosed or clearly described mental health condition that the report names.", "a general reference to \"mental health issues\" with no named condition, a self-reported feeling (for example \"felt depressed\"), and symptoms described only as part of a physical or neurological illness.", "a condition that only the deceased reported, without a diagnosis or clinical description, does not count.", "null. False only if the report explicitly denies it."),
    "mental_health_conditions": ("the specific condition(s) named, using the report's own terminology, when mental_health_flag is true.", "feelings, symptoms and general references to mental health with no named condition.", "use the report's wording; do not substitute a more general or a different term.", "return an empty list."),
    "substance_use_flag": ("an explicit statement of abuse, misuse or dependence on alcohol or other drugs (a diagnosed substance use disorder, addiction, a drug or alcohol problem, or heavy, regular or problematic use), or of smoking or nicotine use.", "social, occasional or experimental use; a medication taken as prescribed; a substance that appears only in toxicology results.", "a substance named without any statement about the deceased's use of it is not counted.", "null. False only if the report explicitly states that the deceased did not misuse such substances."),
    "substances_used": ("the alcohol, tobacco and other drugs that the report names as abused, misused or relied on by the deceased, including tobacco or nicotine where the deceased smoked or used it, when substance_use_flag is true.", "a medication taken as prescribed, and a substance named only in toxicology results.", "if the substance is not named, give the general term the report uses followed by \"(unspecified)\".", "return an empty list."),
    "known_health_issues": ("the deceased's known health conditions, including physical or chronic conditions, injuries, symptoms and acute clinical findings, and the conditions listed under intellectual_disability_conditions and mental_health_conditions, each as a short phrase.", "substance use or dependence itself, and lifestyle behaviours such as smoking or drinking.", "conditions that result from substance use (for example alcohol withdrawal) are health issues and count.", "return an empty list."),
    "custody_served_flag": ("an explicit statement that the deceased served a custodial sentence or was held in custody at any point, including police or correctional custody at the time of death.", "being charged, arrested, or having outstanding warrants alone.", "held in police custody after an arrest counts; an arrest with no statement of being held does not.", "null."),
    "non_english_speaker_flag": ("an explicit statement that the deceased was not a native or fluent English speaker, or had English as an additional language.", "country of birth, name, or cultural background.", "speaking another language is not by itself a statement about English fluency.", "null."),
    "use_of_interpreters_flag": ("an explicit statement that an interpreter was used at any point connected to the case.", "the deceased or family members speaking another language.", "an interpreter used at any stage (medical care, police interview, inquest) counts.", "null (not false)."),
    "preventable_death_flag": ("the coroner explicitly and directly stating that the death was (or was not) preventable.", "the making of recommendations, criticism of care, or a statement that systems could be improved.", "wording such as \"might have been avoided\" is not an explicit statement of preventability.", "null."),
    "domestic_violence_flag": ("a domestic violence related charge or conviction involving the deceased, an apprehended domestic violence order involving the deceased, or a direct statement that the deceased experienced or perpetrated domestic violence.", "relationship conflict, or violence not described as domestic, without such a charge, order or statement.", "a charge, conviction or order for a domestic violence offence counts even if the report gives few details.", "null."),
    "toxicology_results_summary": ("the toxicology findings copied from the report in the coroner's own wording, including the substances detected and the levels found.", "a paraphrase or summary, and post-mortem findings that are not toxicology.", "remove the deceased's name, title and pronouns so that the answer is a subjectless phrase.", "leave empty."),
    "coroners_recommendations_summary": ("the coroner's formal recommendations copied verbatim from the report, including the heading naming the recipient(s) and the original numbering.", "general observations or notes that are not formal recommendations.", "cover every distinct recommendation, keep the numbering, and keep any name of the deceased exactly as written.", "leave empty if the report makes no recommendations."),
}

assert list(CARD_RULES) == list(FIELD_DEFINITIONS), "CARD_RULES and FIELD_DEFINITIONS are out of sync"

# The variables that receive a short card in prompt Z5 (the other variables keep only their Z4 definition). The mental health and intellectual disability flags are not given a short card.
# The variables whose "Counts" line adds something to the definition (an exception, a list of cues or a rule that is not in the definition).
# Only these variables keep the "Counts" line in the full cards; for every other variable the line would repeat the definition.
COUNTS_ADDS = {"sex", "country_of_birth", "intellectual_disability_flag"}

DECISION_CARD_FIELDS = ["sex", "age_at_death", "profession", "known_health_issues", "substance_use_flag", "custody_served_flag",
                        "non_english_speaker_flag", "use_of_interpreters_flag", "preventable_death_flag", "domestic_violence_flag"]


def build_card(name: str, with_definition: bool) -> str:
    """Return the card of one variable as text.

    Args:
        name: The variable name.
        with_definition: Whether the card starts with the definition of the variable (as in prompt Z6). The "Counts" line is then kept only for the variables in COUNTS_ADDS, because for the others it repeats the definition.

    Returns:
        The card: the variable name, the optional definition and the four rules, one per line.
    """
    counts, not_counts, borderline, silent = CARD_RULES[name]
    lines = [name] + ([f"Definition: {FIELD_DEFINITIONS[name]}"] if with_definition else [])
    if not with_definition or name in COUNTS_ADDS:             # with a definition line, "Counts" is kept only where it adds something
        lines.append(f"Counts: {counts}")
    lines += [f"Does not count: {not_counts}", f"Borderline: {borderline}", f"If the report is silent: {silent}"]
    return "\n".join(lines)


def build_decision_cards() -> str:
    """Return the short cards of Z5: only the variables in DECISION_CARD_FIELDS, without the definition line."""
    header = ("Decision cards for the variables that need extra care. Each card repeats the rule from the definitions above and adds what does not count and what to do in borderline cases.")
    closing = ("intellectual_disability_flag and mental_health_flag are not given cards: their definitions are deliberately kept as written above.")
    return "\n\n" + "\n\n".join([header] + [build_card(n, False) for n in DECISION_CARD_FIELDS] + [closing])


def build_variable_cards() -> str:
    """Return the full cards of Z6 and the COT2 prompts: one card, with its definition, for every variable."""
    header = ("Variable cards. Each variable below has a card with its definition, what does not count, the rule for borderline cases and what to return when the report is silent. Where a card has a Counts line, it adds to the definition.")
    return "\n\n" + "\n\n".join([header] + [build_card(n, True) for n in CARD_RULES])


COMPONENTS["decision_cards"] = build_decision_cards()                                  # Z5: Z4 + short cards for ten variables
COMPONENTS["general_rules"] = " ".join([DECEDENT_RULE, BOOLEAN_RULE, EMPTY_RULE, LIST_RULE])      # Z6 and Z7: the general rules without the Z4 definition list
COMPONENTS["variable_cards"] = build_variable_cards()                                  # Z6 and Z7: a card for each of the 28 variables
print("card components added:", [k for k in ("decision_cards", "general_rules", "variable_cards")], "|", len(CARD_RULES), "cards")


## 5. How a prompt reaches the model

Every trial is one call of `client.responses.parse()`, set up as in the OpenAI guide on structured outputs ([developers.openai.com/api/docs/guides/structured-outputs](https://developers.openai.com/api/docs/guides/structured-outputs?api-mode=responses)):

- `input` is a list of **messages**, each with a `role` and `content`.
- `text_format` is the Pydantic schema from Section 2, which constrains the answer to the 28 variables.

The roles used in this notebook:

| Role | Contains |
|---|---|
| **system message** (`"role": "system"`) | The standing instructions: the role, the formatting rules and the definitions (the `COMPONENTS`). |
| **user message** (`"role": "user"`) | The task text (`TASK_CORE`) and the report PDF. |
| **assistant message** (`"role": "assistant"`) | Few-shot only: the correct answer to an example report. |

OpenAI's text-generation guide calls the instruction role `developer`, and the structured-outputs guide uses `system`. The API accepts both, and the `instructions=` parameter has the same effect. This notebook follows the structured-outputs guide.

**The report PDF.** A PDF is added to a user message as `{"type": "input_file", "file_url": "<link>"}`. The API downloads the PDF from the link and passes the model **both the extracted text and an image of each page** ([PDF guide](https://developers.openai.com/api/docs/guides/pdf-files)). One report can therefore use tens of thousands of input tokens.

### Messages sent by each technique

The three layouts below are built by `build_zero_shot_messages()`, `build_few_shot_messages()` and `build_cot_messages()` (Sections 8 to 10), one function per technique.

**ZERO-SHOT**
```
system:    instructions
user:      task text + REPORT (via file_url)
```

**FEW-SHOT**
```
system:    instructions
user:      task text + EXAMPLE 1 (via file_url)
assistant: EXAMPLE 1's correct answer
user:      EXAMPLE 2 (via file_url)
assistant: EXAMPLE 2's correct answer
user:      REPORT (via file_url)
```

**CHAIN-OF-THOUGHT**
```
system:    instructions
user:      task text + reasoning request + REPORT (via file_url)
```

Z0 is the exception: it has no system message and no task text, so the user message holds only the report PDF, and the model receives the variable schema.

`compose()` below turns the switched-on components into the system message text and the user message text. `show_messages()` prints the layout of any message list (the role of each message and each part) without calling the API. Long texts are shortened to 90 characters; `show_messages(messages, max_chars=None)` shows them in full.

In [ ]:
def compose(flags: dict, include_task_core: bool = True) -> tuple[str, str]:
    """Build the system and user text of a prompt from the switched-on components.

    The components that are True in `flags` are joined into the system message; the user message holds the task text. The report PDF is added later, by the build_*_messages functions.

    Args:
        flags: Component name -> True/False, selecting entries of COMPONENTS.
        include_task_core: Whether the user text holds TASK_CORE (otherwise it is empty).

    Returns:
        A (system_text, user_text) tuple.
    """
    system_text = " ".join(text for name, text in COMPONENTS.items() if flags.get(name))
    user_text = TASK_CORE if include_task_core else ""
    return system_text, user_text


def show_messages(messages: list[dict], max_chars: int | None = 90) -> None:
    """Print the layout of a message list without sending anything to the API.

    For each message, the role is printed with each text part (and its word count) or the link of each PDF part. Long texts are shortened unless `max_chars` is None.

    Args:
        messages: The message list, as built by a build_*_messages function.
        max_chars: How many characters of each text to show. Use None to show the whole text, for example the full system message.
    """
    def short(t):
        """Return `t` (quoted, shortened to `max_chars` if set) followed by its word count."""
        shown = t if max_chars is None or len(t) <= max_chars else t[:max_chars] + "..."
        return repr(shown) + f"  [{len(t.split())} words]"
    for m in messages:
        role = f"{m['role']:>9}:"
        if isinstance(m["content"], str):
            print(role, short(m["content"]))
            continue
        for part in m["content"]:
            what = f"input_text  {short(part['text'])}" if part["type"] == "input_text" else f"input_file  {part['file_url']}"
            print(role, what)
            role = " " * 10

## 6. What is saved for each trial, and how prompt changes are recorded

`PromptTrialLog` is the record saved for each trial: the exact prompt text, the report, the model's answer and the number of tokens used. Token counts allow prompts to be compared on their use of tokens (a longer prompt, worked examples and reasoning all increase them). `usage_cached_tokens` is the part of the input that OpenAI served from its prompt cache. `reasoning` is filled only for chain-of-thought trials.

Each record is written to its own folder (`trials/<trial_id>/trial.json`) and appended to one file for the dataset (`audit_log.jsonl`), which is the file read by the evaluation notebook.

**Prompt versions.** Every trial stores the exact text it was sent, and the notebook also records the *versions* of each prompt automatically:

- A prompt name (for example `Z4`) starts at **version 1**.
- If the text sent under that name later differs in any way (system message, task text, chain-of-thought request or worked examples), the next trial is recorded as **version 2**. The full text of each version is stored in `prompt_versions.jsonl`, with the date and a free-text note (`RUN_NOTE`, set before each run).
- Returning to an earlier wording reuses the earlier version number.
- Each trial stores `prompt_version` and `prompt_hash` (a short code of the exact text). The evaluation notebook scores `Z4_v1` and `Z4_v2` as different prompts, so results obtained with different wordings are never combined.

A prompt therefore keeps its name when its wording is edited.

**Request settings.** The settings held constant across prompts are saved with every trial: `model` (the dated model name returned by the API), `reasoning_effort`, `temperature`, `top_p`, `max_output_tokens` and `verbosity` (as set in Section 1; `None` means the setting was not sent and the model's default applied). They are accompanied by `api_reported_settings` (the settings the API reports it applied), `service_tier`, `response_status` and `incomplete_reason` (filled if an answer was cut short). The Responses API has neither a `seed` parameter nor a `system_fingerprint` field, so neither can be recorded.

In [ ]:
class PromptTrialLog(BaseModel):
    """The record saved for one trial: what was sent, what came back, and how many tokens it used."""
    trial_id: str
    dataset: str                            # prompt_development / internal_validation / external_validation / test_report
    report_no: int
    expected_file_number: str               # from the links file
    file_number_match: bool | None = None   # whether the file number read by the model matches the expected one (checks the link)
    technique: str                          # zero_shot / few_shot / chain_of_thought
    n_shots: int = 0                        # number of worked examples (few-shot only)
    prompt_id: str                          # the name of the prompt, for example Z4 or FS2
    prompt_version: int = 0                 # 1, 2, ... = the wording of this prompt name (0 = test report, not versioned)
    prompt_hash: str = ""                   # short code of the exact prompt text; same code = same text
    repeat_index: int
    test_mode: bool = False                 # True = test run; never part of the study
    factors: dict                           # which components were switched on
    include_task_core: bool
    system_text: str                        # the exact text of the system message ("" = none)
    user_text: str                          # the exact text of the user message, before the PDF ("" = none)
    example_urls: list[str] = []            # the PDFs used as worked examples (few-shot only)
    file_url: str                           # the report PDF
    notes: str = ""
    response_id: str
    created_at: str
    completed_at: str
    latency_seconds: float
    model: str | None
    reasoning_effort: str | None = None     # request settings, as set in Section 1; None = not sent (the model's default)
    temperature: float | None = None
    top_p: float | None = None
    max_output_tokens: int | None = None
    verbosity: str | None = None
    api_reported_settings: dict = {}        # what the API reports it applied: temperature, top_p, max_output_tokens, reasoning_effort, verbosity
    service_tier: str | None = None         # the service tier that processed the request, as returned by the API
    response_status: str | None = None      # completed / incomplete / failed
    incomplete_reason: str | None = None    # why an answer was cut short (for example max_output_tokens)
    usage_input_tokens: int                 # includes the PDF(s)
    usage_cached_tokens: int = 0            # part of the input served from the prompt cache
    usage_output_tokens: int                # includes any hidden reasoning tokens
    usage_reasoning_tokens: int = 0
    usage_total_tokens: int
    output: dict                            # the 28 extracted variables
    reasoning: str | None = None            # chain-of-thought text (chain-of-thought only)

The cell below maintains the list of prompt versions in `prompt_versions.jsonl` (in the results folder). `resolve_prompt_version()` is called once for each trial: it compares the exact prompt text with the versions already recorded for that prompt name, and either returns the matching version or records a new one. A prompt text that is new in internal or external validation produces a warning, because only prompts run in prompt development may be run there. `show_prompt_versions()` lists all versions, and `show_prompt_version("Z4", 1)` prints the full text of one version (for example, to restore an earlier wording).

In [ ]:
PROMPT_VERSIONS_FILE = RESULTS_ROOT / "prompt_versions.jsonl"


def prompt_fingerprint(technique: str, system_text: str, user_text: str, examples: list[dict], schema_name: str | None = None) -> str:
    """Return a short code that identifies the exact prompt.

    The same technique, text and worked examples always give the same code, so the code shows whether a prompt has changed between runs.

    Args:
        technique: The prompting technique, for example "zero_shot".
        system_text: The system message.
        user_text: The user message (before the PDF).
        examples: The worked examples (file_url and ideal_output of each, and the note if there is one).
        schema_name: The name of the response schema, given only when it is not the default shape (ExtractionWithReasoning for chain of thought, the plain extraction otherwise). Left out of the code when None, so earlier prompts keep their code.

    Returns:
        The first 10 hexadecimal characters of the SHA-256 hash of the prompt.
    """
    content = {"technique": technique, "system": system_text, "user": user_text,
               "examples": [{k: e[k] for k in ("file_url", "ideal_output", "note") if k in e} for e in examples]}
    if schema_name:
        content["schema"] = schema_name
    return hashlib.sha256(json.dumps(content, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()[:10]


def read_prompt_versions() -> list[dict]:
    """Read every recorded prompt version from PROMPT_VERSIONS_FILE.

    Returns:
        One dict per recorded version, in the order recorded; an empty list if the file does not exist yet.
    """
    if not PROMPT_VERSIONS_FILE.exists():
        return []
    return [json.loads(line) for line in open(PROMPT_VERSIONS_FILE) if line.strip()]


def resolve_prompt_version(prompt_id: str, technique: str, system_text: str, user_text: str,
                           examples: list[dict], dataset: str, note: str = "", schema_name: str | None = None) -> tuple[int, str]:
    """Return the version of the prompt that was just sent, recording it if it is new.

    A prompt text seen before keeps its version. A new text gets the next version number and is appended to PROMPT_VERSIONS_FILE, except on the test report, where no version is created (version 0).

    Args:
        prompt_id: The prompt name, for example "Z4".
        technique: The prompting technique.
        system_text: The system message.
        user_text: The user message (before the PDF).
        examples: The worked examples (empty for zero-shot and chain-of-thought).
        dataset: The set the trial belongs to.
        note: Free-text note stored with a newly recorded version.
        schema_name: The response schema, if it is not the default shape (see prompt_fingerprint).

    Returns:
        A (version, prompt_hash) tuple.
    """
    prompt_hash = prompt_fingerprint(technique, system_text, user_text, examples, schema_name)
    known = [v for v in read_prompt_versions() if v["prompt_id"] == prompt_id]
    for v in known:
        if v["prompt_hash"] == prompt_hash:
            return v["version"], prompt_hash                          # this exact text has been used before
    if dataset == "test_report":
        return 0, prompt_hash                                        # test runs never create versions
    version = max((v["version"] for v in known), default=0) + 1
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    with open(PROMPT_VERSIONS_FILE, "a") as f:
        f.write(json.dumps(dict(prompt_id=prompt_id, version=version, prompt_hash=prompt_hash, technique=technique,
                                n_shots=len(examples), first_used=str(datetime.now()), first_dataset=dataset, note=note,
                                system_text=system_text, user_text=user_text, examples=examples,
                                **({"schema": schema_name} if schema_name else {})), ensure_ascii=False) + "\n")
    if known:
        print(f"  NOTE: the text of prompt {prompt_id} differs from its earlier version(s): recorded as {prompt_id}_v{version}.")
    if dataset in ("internal_validation", "external_validation"):
        print(f"  WARNING: {prompt_id}_v{version} is a new wording that was not run in prompt development. Use the wording that was chosen (see show_prompt_versions()).")
    return version, prompt_hash


def show_prompt_versions() -> pd.DataFrame:
    """Return a table of the recorded prompt versions, one row per version.

    Returns:
        A DataFrame with the prompt name, version, technique, number of examples, word count, hash, first set and first use of each version; empty if nothing has been recorded.
    """
    versions = pd.DataFrame(read_prompt_versions())
    if versions.empty:
        return versions
    versions["words"] = versions["system_text"].str.split().str.len() + versions["user_text"].str.split().str.len()
    return versions[["prompt_id", "version", "technique", "n_shots", "words", "prompt_hash", "first_dataset", "first_used", "note"]]


def show_prompt_version(prompt_id: str, version: int) -> None:
    """Print the full text of one recorded prompt version (to restore an earlier wording).

    Args:
        prompt_id: The prompt name, for example "Z4".
        version: The version number.

    Raises:
        StopIteration: If that version has not been recorded.
    """
    v = next(v for v in read_prompt_versions() if v["prompt_id"] == prompt_id and v["version"] == version)
    print(f"{prompt_id}_v{version} ({v['technique']}, {v['n_shots']} example(s), first used {v['first_used']}, note: {v['note']!r})")
    print("\n--- system message ---\n" + (v["system_text"] or "(none)"))
    print("\n--- user message (before the PDF) ---\n" + (v["user_text"] or "(none)"))
    for i, e in enumerate(v["examples"], 1):
        print(f"\n--- worked example {i}: {e['file_url']} ---\n" + json.dumps(e["ideal_output"], ensure_ascii=False)[:300] + " ...")

In [ ]:
def dataset_dir(dataset: str) -> Path:
    """Return the results folder of one set, creating it if needed.

    Args:
        dataset: Name of the set.

    Returns:
        The path RESULTS_ROOT / dataset.
    """
    d = RESULTS_ROOT / dataset
    d.mkdir(parents=True, exist_ok=True)
    return d


def save_trial(dataset: str, log_entry: PromptTrialLog) -> None:
    """Save one trial to its own folder and append it to the set's audit log.

    Writes `trials/<trial_id>/trial.json` and adds one line to `audit_log.jsonl` in the set's results folder.

    Args:
        dataset: Name of the set.
        log_entry: The trial record to save.
    """
    t_dir = dataset_dir(dataset) / "trials" / log_entry.trial_id
    t_dir.mkdir(parents=True, exist_ok=True)
    (t_dir / "trial.json").write_text(log_entry.model_dump_json(indent=2))
    with open(dataset_dir(dataset) / "audit_log.jsonl", "a") as f:
        f.write(log_entry.model_dump_json() + "\n")

## 7. Helper functions shared by the three techniques

Each technique has its own trial function (Sections 8 to 10) with the same four steps: **check the dataset, build the messages, call the model, log the trial.** Three of these steps are identical for every technique and are defined once here:

- `check_trial_allowed()` checks that the report may be run (the correct dataset, an allowed dataset and a link present). Nothing is sent to the API if the check fails.
- `call_model()` makes the API call (`client.responses.parse()`) with the model and the output format.
- `log_trial()` converts the response into a `PromptTrialLog` (including the prompt version), saves it and returns it. `print_trial()` prints one line per trial.

The messages are the only step that differs between techniques, so they are built in separate functions in Sections 8 to 10.

In [ ]:
def check_trial_allowed(report_no: int, dataset: str) -> dict:
    """Check, before anything is sent to the API, that this report may be run now.

    Args:
        report_no: The report number.
        dataset: The set the trial is meant to belong to.

    Returns:
        The report's entry in REPORTS.

    Raises:
        KeyError: If the report is not in the links file.
        PermissionError: If the set is not in ALLOWED_DATASETS, the report belongs to another set, the report has been withdrawn, or the report has no link.
    """
    if report_no not in REPORTS:
        raise KeyError(f"Report {report_no} is not in {LINKS_FILE}.")
    report = REPORTS[report_no]
    if dataset not in ALLOWED_DATASETS:
        raise PermissionError(f"The set '{dataset}' is not allowed to run yet (ALLOWED_DATASETS = {sorted(ALLOWED_DATASETS)}).")
    if report["dataset"] != dataset:
        raise PermissionError(f"Report {report_no} belongs to '{report['dataset']}', not '{dataset}'.")
    if report["status"] == "withdrawn":
        raise PermissionError(f"Report {report_no} has been withdrawn from the court website and is not run.")
    if not report["file_url"]:
        raise PermissionError(f"Report {report_no} has no link in {LINKS_FILE}.")
    return report

In [ ]:
def settings_in_use() -> dict:
    """Return the request settings that are set in Section 1 (those that are not None).

    The names are the ones used in PromptTrialLog.

    Returns:
        A dict of setting name -> value.
    """
    chosen = {"reasoning_effort": REASONING_EFFORT, "temperature": TEMPERATURE, "top_p": TOP_P,
              "max_output_tokens": MAX_OUTPUT_TOKENS, "verbosity": VERBOSITY}
    return {name: value for name, value in chosen.items() if value is not None}


def api_reported_settings(response) -> dict:
    """Return the settings that the API reports it applied to a response.

    Args:
        response: The API response from call_model.

    Returns:
        A dict with temperature, top_p, max_output_tokens, reasoning_effort and verbosity (None where the response does not report a value).
    """
    reasoning = getattr(response, "reasoning", None)
    text_config = getattr(response, "text", None)
    return {"temperature": getattr(response, "temperature", None), "top_p": getattr(response, "top_p", None),
            "max_output_tokens": getattr(response, "max_output_tokens", None),
            "reasoning_effort": getattr(reasoning, "effort", None), "verbosity": getattr(text_config, "verbosity", None)}


def call_model(messages: list[dict], text_format):
    """Make one API call and time it, with the request settings from Section 1.

    Args:
        messages: The message list to send.
        text_format: The schema the answer must follow.

    Returns:
        A (response, seconds) tuple.
    """
    settings = settings_in_use()
    extra = {name: settings[name] for name in ("temperature", "top_p", "max_output_tokens") if name in settings}
    if "reasoning_effort" in settings:
        extra["reasoning"] = {"effort": settings["reasoning_effort"]}
    if "verbosity" in settings:
        extra["text"] = {"verbosity": settings["verbosity"]}
    start = time.time()
    response = client.responses.parse(model=MODEL, input=messages, text_format=text_format, **extra)
    return response, time.time() - start


def log_trial(response, seconds: float, *, report_no: int, dataset: str, technique: str, prompt_id: str, repeat_index: int,
              flags: dict, include_task_core: bool, system_text: str, user_text: str,
              examples: list[dict] | None = None, use_cot: bool = False, notes: str = "", schema_name: str | None = None) -> dict:
    """Turn an API response into a saved trial record.

    The response is parsed into a PromptTrialLog, the prompt version is resolved (and recorded if new), and the trial is saved. A warning is printed if the file number read by the model differs from the expected one.

    Args:
        response: The API response from call_model.
        seconds: The duration of the call.
        report_no: The report number.
        dataset: The set the trial belongs to.
        technique: The prompting technique.
        prompt_id: The prompt name.
        repeat_index: The repeat number of this prompt on this report.
        flags: The prompt components that were switched on.
        include_task_core: Whether the task text was included.
        system_text: The system message that was sent.
        user_text: The user message that was sent (before the PDF).
        examples: The worked examples, for few-shot prompts.
        use_cot: Whether the answer holds `reasoning` and `extraction` parts.
        notes: Free-text note stored with the trial.
        schema_name: The response schema, if it is not the default shape; it is part of the prompt record.

    Returns:
        The saved record, as a dict.
    """
    examples = examples or []
    report = REPORTS[report_no]
    parsed = json.loads(response.output_text)
    if use_cot:                 # reasoning text, or the evidence list of the structured variant (kept as JSON text)
        reasoning = parsed["reasoning"] if "reasoning" in parsed else json.dumps(parsed["evidence"], ensure_ascii=False)
        output = parsed["extraction"]
    else:
        reasoning, output = None, parsed
    usage = response.usage
    cached = getattr(getattr(usage, "input_tokens_details", None), "cached_tokens", 0) or 0
    reasoning_tokens = getattr(getattr(usage, "output_tokens_details", None), "reasoning_tokens", 0) or 0
    version, prompt_hash = resolve_prompt_version(prompt_id, technique, system_text, user_text, examples, dataset, notes, schema_name)

    log_entry = PromptTrialLog(
        trial_id=f"{datetime.now().strftime('%Y%m%dT%H%M%S%f')}_{response.id[-8:]}",
        dataset=dataset, report_no=report_no, expected_file_number=report["file_number"],
        file_number_match=(str(output.get("file_number", "")).strip() == report["file_number"]),
        technique=technique, n_shots=len(examples), prompt_id=prompt_id, prompt_version=version, prompt_hash=prompt_hash,
        repeat_index=repeat_index, test_mode=TEST_MODE, factors=flags, include_task_core=include_task_core,
        system_text=system_text, user_text=user_text, example_urls=[e["file_url"] for e in examples],
        file_url=report["file_url"], notes=notes, response_id=response.id,
        created_at=str(datetime.fromtimestamp(response.created_at)),
        completed_at=str(datetime.fromtimestamp(response.completed_at)) if getattr(response, "completed_at", None) else "",
        latency_seconds=round(seconds, 4), model=response.model, **settings_in_use(),
        usage_input_tokens=usage.input_tokens, usage_cached_tokens=cached, usage_output_tokens=usage.output_tokens,
        usage_reasoning_tokens=reasoning_tokens, usage_total_tokens=usage.total_tokens,
        output=output, reasoning=reasoning, api_reported_settings=api_reported_settings(response),
        service_tier=getattr(response, "service_tier", None), response_status=getattr(response, "status", None),
        incomplete_reason=getattr(getattr(response, "incomplete_details", None), "reason", None),
    )
    save_trial(dataset, log_entry)
    if not log_entry.file_number_match:
        print(f"  WARNING: report {report_no}: the model read file number {output.get('file_number')!r}, expected {report['file_number']!r}. Check the link.")
    if log_entry.response_status not in (None, "completed"):
        print(f"  WARNING: report {report_no}: the response status is {log_entry.response_status!r} ({log_entry.incomplete_reason}); the answer may be cut short.")
    return log_entry.model_dump()


def print_trial(record: dict) -> None:
    """Print a one-line summary of a saved trial (prompt, repeat, tokens and trial id).

    Args:
        record: The trial record returned by a run_*_trial function.
    """
    label = f"{record['prompt_id']}_v{record['prompt_version']}" if record["prompt_version"] else record["prompt_id"]
    print(f"report {record['report_no']} | {label} repeat {record['repeat_index']}: {record['usage_input_tokens']} in ({record['usage_cached_tokens']} cached) / {record['usage_output_tokens']} out tokens -> {record['trial_id']}")

## 8. Zero-shot

### 8.1 The messages

Zero-shot sends a system message (the instructions, omitted when a prompt has none) and one user message holding the task text (also omitted when a prompt has none) and the report PDF. The report is added as an `input_file` with its `file_url`.

In [ ]:
def build_zero_shot_messages(system_text: str, user_text: str, file_url: str) -> list[dict]:
    """Build the message list of a zero-shot prompt.

    Layout:
        system:  the instructions (left out when empty)
        user:    the task text (left out when empty) + the REPORT, as a PDF link

    Args:
        system_text: The system message.
        user_text: The task text.
        file_url: The link to the report PDF.

    Returns:
        The message list.
    """
    messages = []
    if system_text:
        messages.append({"role": "system", "content": system_text})

    user_content = []
    if user_text:
        user_content.append({"type": "input_text", "text": user_text})
    user_content.append({"type": "input_file", "file_url": file_url})
    messages.append({"role": "user", "content": user_content})
    return messages

### 8.2 The zero-shot trial function

Four steps: check the dataset, build the messages, call the model, log the trial.

In [ ]:
def run_zero_shot_trial(report_no: int, prompt_id: str, flags: dict, include_task_core: bool = True, *,
                        dataset: str, repeat_index: int = 1, notes: str = "") -> dict:
    """Run one zero-shot call on one report and save it.

    Args:
        report_no: The report number.
        prompt_id: The prompt name.
        flags: The prompt components to switch on.
        include_task_core: Whether to include the task text.
        dataset: The set the trial belongs to.
        repeat_index: The repeat number of this prompt on this report.
        notes: Free-text note stored with the trial.

    Returns:
        The saved record, as a dict.
    """
    report = check_trial_allowed(report_no, dataset)                                         # 1. check
    system_text, user_text = compose(flags, include_task_core)
    messages = build_zero_shot_messages(system_text, user_text, report["file_url"])          # 2. messages
    response, seconds = call_model(messages, VariablesToExtractExtended)                     # 3. call
    return log_trial(response, seconds, report_no=report_no, dataset=dataset, technique="zero_shot",   # 4. log
                     prompt_id=prompt_id, repeat_index=repeat_index, flags=flags, include_task_core=include_task_core,
                     system_text=system_text, user_text=user_text, notes=notes)

### 8.3 The zero-shot prompts

Each prompt adds one type of content, so that the effect of each component can be measured (dissertation, Table 6 and Appendix G, Table G1). The system message holds the components; the user message holds the task text and the report.

| Prompt | System message | User message |
|---|---|---|
| **Z0** | none | the report only (no instructions: the model receives the schema and the PDF) |
| **Z1** | none | task text + report |
| **Z2** | role, date format, acronyms, level of detail | task text + report |
| **Z3** | Z2 + definitions of a mental health condition and an intellectual disability | task text + report |
| **Z4** | Z3 + the definition of every variable and the general rules | task text + report |
| **Z5** | Z4 + short decision cards (what counts, what does not count, borderline cases, report silent) for ten variables | task text + report |
| **Z6** | Z3 + the general rules + a decision card for each of the 28 variables (the definition is the first line; "Counts" only where it adds to the definition), in place of the Z4 definition list | task text + report |
| **Z7** | the general rules + a decision card for each variable only (no role, date, acronym, level-of-detail or clinical-definition components) | task text + report |

Z1 is not an empty prompt: the task text instructs the model what to extract, to leave missing information empty and to give no commentary.

**Changing a prompt.** A change to the text in Section 4 or to the components of a prompt below is recorded automatically as a new version of that prompt (Section 6). A new prompt is added as a new entry of `ZERO_SHOT_PROMPTS`.

In [ ]:
Z2_FLAGS = {"role_definition": True, "date_format_instruction": True, "acronym_instruction": True, "detail_level_instruction": True}
Z3_FLAGS = {**Z2_FLAGS, "mhc_id_icd_definitions": True}
Z4_FLAGS = {**Z3_FLAGS, "variable_definitions": True}
Z5_FLAGS = {**Z4_FLAGS, "decision_cards": True}                                  # Z4 + short cards for ten variables
Z6_FLAGS = {**Z3_FLAGS, "general_rules": True, "variable_cards": True}           # Z3 + the general rules + a card for every variable (no Z4 definition list)
Z7_FLAGS = {"general_rules": True, "variable_cards": True}                       # the general rules + a card for every variable only (no role, format or clinical-definition components)

# Prompts that are left out of the run lists (Run All skips them).
#   ALREADY_RUN: prompts whose prompt development results are stored; they are not run again (remove a name to run it again).
#   NOT_YET_RUN: defined but not yet run (remove a name to run it).
ALREADY_RUN = {"Z0", "Z1", "Z2", "Z3", "Z4", "Z5", "Z6",
               "FS1", "FS2", "FS1_notes", "FS2_notes", "FS2_pairs",
               "COT1", "COT1_struct", "COT1_evidence", "COT2_B", "COT2_C"}   # internal validation runs only the best prompt of each technique: Z7, FS2_pairs_Z6 and COT2_A
NOT_YET_RUN = set()
# External validation runs only the prompt selected at internal validation (Z7_v2).
NOT_IN_EXTERNAL_VALIDATION = {"FS2_pairs_Z6", "COT2_A"}
SKIP = ALREADY_RUN | NOT_YET_RUN | NOT_IN_EXTERNAL_VALIDATION

# prompt_id -> which components are switched on, and whether the task text is included
ZERO_SHOT_PROMPTS = {
    "Z0": {"flags": {},       "include_task_core": False},
    "Z1": {"flags": {},       "include_task_core": True},
    "Z2": {"flags": Z2_FLAGS, "include_task_core": True},
    "Z3": {"flags": Z3_FLAGS, "include_task_core": True},
    "Z4": {"flags": Z4_FLAGS, "include_task_core": True},
    # prompts with decision cards
    "Z5": {"flags": Z5_FLAGS, "include_task_core": True},    # Z4 + short decision cards for ten variables
    "Z6": {"flags": Z6_FLAGS, "include_task_core": True},    # cards for all 28 variables instead of the Z4 definition list
    # decision cards without the role, format and clinical components
    "Z7": {"flags": Z7_FLAGS, "include_task_core": True},    # task text + general rules + the 28 cards
}
ZERO_SHOT_TO_RUN = [p for p in ZERO_SHOT_PROMPTS if p not in SKIP]
if TEST_MODE:        # one prompt per technique is sufficient to test the pipeline and the agreement metrics
    ZERO_SHOT_TO_RUN = ["Z4"]
print("zero-shot prompts to run:", ZERO_SHOT_TO_RUN)

**Preview.** The layout of the messages for Z4, without calling the API:

In [ ]:
_p = ZERO_SHOT_PROMPTS["Z4"]
_system, _user = compose(_p["flags"], _p["include_task_core"])
show_messages(build_zero_shot_messages(_system, _user, REPORTS[TEST_REPORT_NO]["file_url"]))
# To see the full texts, call show_messages(..., max_chars=None)


**Preview of Z5, Z6 and Z7.** Long texts are shortened; `show_messages(..., max_chars=None)` shows the whole text.

In [ ]:
for _id in ["Z5", "Z6", "Z7"]:
    _p = ZERO_SHOT_PROMPTS[_id]
    _system, _user = compose(_p["flags"], _p["include_task_core"])
    print(f"--- {_id}")
    show_messages(build_zero_shot_messages(_system, _user, REPORTS[TEST_REPORT_NO]["file_url"]))


### 8.4 Smoke test

One call (prompt Z4) on the test report, run in test mode (`TEST_MODE = True`) after any change of `MODEL`. It confirms that the model name is valid and that a PDF link is accepted, and shows how many tokens a call uses. The record is saved in the `test_report` folder and is not part of any study dataset.

With `TEST_MODE = False` the cell prints a note and does nothing, so that *Run All* can be used for the study runs.

In [ ]:
if TEST_MODE:      # smoke tests run in test mode only (they use the test report); skipped in study runs
    _p = ZERO_SHOT_PROMPTS["Z4"]
    record = run_zero_shot_trial(TEST_REPORT_NO, "Z4", _p["flags"], _p["include_task_core"], dataset="test_report", notes="smoke test")
    print("model:", record["model"], "| file number matched:", record["file_number_match"])
    print(f"tokens: {record['usage_input_tokens']} in ({record['usage_cached_tokens']} cached), {record['usage_output_tokens']} out, {record['usage_total_tokens']} total | {record['latency_seconds']} s")
else:
    print("Smoke test skipped: TEST_MODE is False (the smoke tests belong to test mode).")

### 8.5 Run the zero-shot prompts

The settings come from Section 3: `DATASET`, `TARGET_REPORTS` and `N_REPEATS` (5 repeats in prompt development and internal validation, 1 in external validation). Every report is run with every prompt in `ZERO_SHOT_TO_RUN` (in test mode only Z4, set in Section 8.3; that cell must be run again after `TEST_MODE` is changed), and each combination is repeated `N_REPEATS` times. `RUN_NOTE` is an optional free-text note saved with every trial and with any new prompt version (for example, what was changed and why).

In [ ]:
RUN_NOTE = "External validation: Z7_v2 (best prompt on internal validation), run unchanged"        # optional note, for example "Z4: added a rule about smoking"

for report_no in TARGET_REPORTS:
    for prompt_id in ZERO_SHOT_TO_RUN:
        prompt = ZERO_SHOT_PROMPTS[prompt_id]
        for repeat_index in range(1, N_REPEATS + 1):
            record = run_zero_shot_trial(report_no, prompt_id, prompt["flags"], prompt["include_task_core"],
                                         dataset=DATASET, repeat_index=repeat_index, notes=RUN_NOTE)
            print_trial(record)

## 9. Few-shot

### 9.1 How it works

**Several PDFs in one call.** A `file_url` is a web link to a PDF. In every few-shot call the API downloads each linked PDF and gives the model its text and page images, as for the report being extracted. A two-shot call therefore contains **three PDFs**, in this order:

1. user message: the task text and **example report 1**
2. assistant message: the correct answer to example 1 (the dictionary below)
3. user message: **example report 2**
4. assistant message: the correct answer to example 2
5. user message: **the report to extract**, which the model answers

The examples are presented as an earlier part of the conversation, and the model is then asked to do the same for the last report. The examples are sent again in every call, so a one-shot call uses roughly twice, and a two-shot call roughly three times, the input tokens of a zero-shot call (the combined size of the PDFs in one request must remain under 50 MB, which these reports do).

**The examples.** `FEW_SHOT_EXAMPLES` holds the two example reports (EX1 and EX2 of the sheet `few_shot_examples` in the reference standard workbook), each as the link to its PDF and the correct answer in the same form as the model's output (the 28 variables). Neither example is one of the 50 study reports (dissertation, Appendix E.4).

In [ ]:
FEW_SHOT_EXAMPLES = [
    {
        # EX1: file 2021/196736 (annotated from this report; not one of the 50 study reports; corrected after the first run, see the note above Section 9.6)
        "file_url": "https://coroners.nsw.gov.au/documents/findings/2025/Inquest_into_the_Death_of_Frank_Dwayne_COLEMAN.pdf",
        "ideal_output": {
            "file_number": "2021/196736",
            "sex": "Male",
            "age_at_death": "43",
            "date_of_findings": "22 October 2025",
            "date_of_death": "between 8:00pm on 7 July 2021 and 5:24am on 8 July 2021",
            "place_of_death": "Metropolitan Special Programs Centre at Long Bay Correctional Complex",
            "definitive_cause_of_death": "methadone toxicity",
            "potential_causes_of_death": [],
            "manner_of_death": "Death occurred after consuming methadone that was supplied by another inmate",
            "country_of_birth": "Australia",
            "place_of_residence": "",
            "profession": "",
            "level_of_education": "Ceased university shortly after joining",
            "medications": ["Aripiprazole", "Atorvastatin", "Bisoprolol", "Entresto (sacubitril / valsartan)", "Escitalopram", "Europagliflozin", "Metformin", "Rabeprazole", "Spironolactone", "Zyprexa"],
            "intellectual_disability_flag": None,
            "intellectual_disability_conditions": [],
            "mental_health_flag": True,
            "mental_health_conditions": ["chronic schizophrenia", "paranoid schizophrenia"],
            "substance_use_flag": True,
            "substances_used": ["alcohol", "tobacco", "cannabis", "heroin", "methamphetamine"],
            "known_health_issues": ["chronic schizophrenia", "paranoid schizophrenia", "cardiomyopathy", "congestive cardiac failure", "gastro-oesophageal reflux disease (GORD)", "hepatitis B", "hepatitis C", "hepatitis D", "hypercholesterolaemia", "hypertension", "thiamine deficiency", "myocardial infarction", "cirrhosis of liver"],
            "custody_served_flag": True,
            "non_english_speaker_flag": None,
            "use_of_interpreters_flag": None,
            "preventable_death_flag": None,
            "domestic_violence_flag": True,
            "toxicology_results_summary": "Toxicological analysis detected methadone in the blood at 0.41 mg/L, a level within the reported lethal range. Aripiprazole was detected at a supratherapeutic level, along with therapeutic type concentrations of citalopram, metformin and olanzapine. Low levels of atorvastatin and bisoprolol were also detected.",
            "coroners_recommendations_summary": "To the Chief Executive, Justice Health and Forensic Mental Health Network NSW (Justice Health)/ the Commissioner, Corrective Services NSW\n1. Justice Health, in coordination with Corrective Services NSW, review the models of care and resourcing of Drug and Alcohol assessments for the commencement of opioid agonist therapy (OAT), with a view to increasing access and capacity, and\na. Produce a joint report estimating unmet need in the public jail system and outlining the optimum wait time\nb. Produce a joint mapping report indicating what resources would be necessary to achieve the optimum wait time (including staffing and infrastructure needs)\n\n2. Justice Health, in coordination with Corrective Services NSW:\na. assess the viability of Drug and Alcohol reviews being conducted via telehealth, including in-cell, and if deemed viable\nb. trial providing Drug and Alcohol reviews by in-cell telehealth",
        },
    },
    {
        # EX2: file 2023/74097 (annotated from this report; not one of the 50 study reports; corrected after the first run, see the note above Section 9.6)
        "file_url": "https://coroners.nsw.gov.au/documents/findings/2024/FURNER_Brett_-_Findings.pdf",
        "ideal_output": {
            "file_number": "2023/74097",
            "sex": "Male",
            "age_at_death": "54",
            "date_of_findings": "25 June 2024",
            "date_of_death": "5 March 2023",
            "place_of_death": "Goulburn Base Hospital, Goulburn NSW 2580",
            "definitive_cause_of_death": "acute myocardial infarction due to coronary artery atherosclerotic disease",
            "potential_causes_of_death": [],
            "manner_of_death": "Died of natural causes whilst in the lawful custody of the New South Wales Police Force",
            "country_of_birth": "",
            "place_of_residence": "",
            "profession": "",
            "level_of_education": "theological college",
            "medications": [],
            "intellectual_disability_flag": None,
            "intellectual_disability_conditions": [],
            "mental_health_flag": True,
            "mental_health_conditions": ["depression", "bipolar disorder", "delusional disorder"],
            "substance_use_flag": True,
            "substances_used": ["alcohol", "tobacco"],
            "known_health_issues": ["depression", "bipolar disorder", "delusional disorder", "easily angered", "delusional behaviour", "paranoid behaviour", "struck by a vehicle", "triple vessel coronary artery disease and severe left ventricular dysfunction", "hypertension", "alcohol withdrawal", "critical stenosis of the left anterior descending coronary artery", "elevated blood pressure", "abnormal liver function", "myocardial infarction", "back injury"],
            "custody_served_flag": True,
            "non_english_speaker_flag": None,
            "use_of_interpreters_flag": None,
            "preventable_death_flag": None,
            "domestic_violence_flag": None,
            "toxicology_results_summary": "toxicological analysis identified significant intoxication at the time of death (blood alcohol level of 0.171 g/100mL), with low levels of diazepam and its active metabolite",
            "coroners_recommendations_summary": "",
        },
    },
]

In [ ]:
# Checks: each answer must fit the schema, and no example may be one of the 50 study reports.
_study_file_numbers = {r["file_number"] for n, r in REPORTS.items() if n != TEST_REPORT_NO}
for _ex in FEW_SHOT_EXAMPLES:
    VariablesToExtractExtended(**_ex["ideal_output"])                      # raises an error if a variable is missing or mistyped
    assert _ex["ideal_output"]["file_number"] not in _study_file_numbers, "an example must not be a study report"
print(f"{len(FEW_SHOT_EXAMPLES)} examples OK:", [e["ideal_output"]["file_number"] for e in FEW_SHOT_EXAMPLES])

### 9.2 The messages

The system message holds the instructions. The task text appears once, in the first user message, with the first worked example. Each example is a user message (the example PDF) followed by an assistant message (its correct answer, as JSON). The report to extract is always the last user message.

In [ ]:
def build_few_shot_messages(system_text: str, user_text: str, examples: list[dict], file_url: str) -> list[dict]:
    """Build the message list of a few-shot prompt.

    Layout:
        system:     the instructions (left out when empty)
        user:       the task text + EXAMPLE 1, as a PDF link
        assistant:  EXAMPLE 1's correct answer
        user:       [note on example 1] + EXAMPLE 2, as a PDF link
        assistant:  EXAMPLE 2's correct answer
        user:       [note on example 2] + the REPORT to extract, as a PDF link

    A note is optional text that explains the hard decisions of an example. It is
    given as the key `note` of the example and is placed in the next user message,
    before the next PDF, so the assistant message stays the bare answer.

    Args:
        system_text: The system message.
        user_text: The task text (placed once, before the first example).
        examples: The worked examples, each with `file_url`, `ideal_output` and optionally `note`.
        file_url: The link to the report PDF.

    Returns:
        The message list.
    """
    messages = []
    if system_text:
        messages.append({"role": "system", "content": system_text})

    for i, example in enumerate(examples):
        user_content = []
        if i == 0 and user_text:
            user_content.append({"type": "input_text", "text": user_text})                 # the task text appears once, first
        if i > 0 and examples[i - 1].get("note"):
            user_content.append({"type": "input_text", "text": examples[i - 1]["note"]})  # note on the previous example
        user_content.append({"type": "input_file", "file_url": example["file_url"]})
        messages.append({"role": "user", "content": user_content})
        messages.append({"role": "assistant", "content": json.dumps(example["ideal_output"])})

    final_content = []
    if examples and examples[-1].get("note"):
        final_content.append({"type": "input_text", "text": examples[-1]["note"]})         # note on the last example
    final_content.append({"type": "input_file", "file_url": file_url})
    messages.append({"role": "user", "content": final_content})
    return messages


### 9.3 The few-shot trial function

In [ ]:
def run_few_shot_trial(report_no: int, prompt_id: str, flags: dict, examples: list[dict], include_task_core: bool = True, *,
                       dataset: str, repeat_index: int = 1, notes: str = "") -> dict:
    """Run one few-shot call on one report, with the given worked examples, and save it.

    Args:
        report_no: The report number.
        prompt_id: The prompt name.
        flags: The prompt components to switch on.
        examples: The worked examples (at least one).
        include_task_core: Whether to include the task text.
        dataset: The set the trial belongs to.
        repeat_index: The repeat number of this prompt on this report.
        notes: Free-text note stored with the trial.

    Returns:
        The saved record, as a dict.

    Raises:
        AssertionError: If no worked example is given.
    """
    assert examples, "few-shot needs at least one worked example"
    report = check_trial_allowed(report_no, dataset)                                         # 1. check
    system_text, user_text = compose(flags, include_task_core)
    messages = build_few_shot_messages(system_text, user_text, examples, report["file_url"]) # 2. messages
    response, seconds = call_model(messages, VariablesToExtractExtended)                     # 3. call
    return log_trial(response, seconds, report_no=report_no, dataset=dataset, technique="few_shot",     # 4. log
                     prompt_id=prompt_id, repeat_index=repeat_index, flags=flags, include_task_core=include_task_core,
                     system_text=system_text, user_text=user_text, examples=examples, notes=notes)

### 9.4 The few-shot prompts

FS1 gives one worked example and FS2 gives two (dissertation, Table 6 and Appendix G, Table G2). Each prompt has its own instructions (`flags`): the Z4 instructions, or the Z6 decision cards for FS2_pairs_Z6. A change to the instructions, the task text or a worked example is recorded automatically as a new version of that prompt (Section 6).

| Prompt | Instructions | Worked examples | Added |
|---|---|---|---|
| **FS1** | Z4 | 1 | nothing |
| **FS2** | Z4 | 2 | nothing |
| **FS1_notes** | Z4 | 1 | a short note on the example (the reason for each difficult decision) |
| **FS2_notes** | Z4 | 2 | a short note on each example |
| **FS2_pairs** | Z4 | 2 | the notes and contrastive pairs (a wrong and a right answer) |
| **FS2_pairs_Z6** | Z6 | 2 | the same notes and pairs as FS2_pairs; only the instructions differ (Z6 decision cards in place of the Z4 definitions) |

The notes and contrastive pairs are in `EXAMPLE_NOTES` and `CONTRASTIVE_PAIRS` below. The worked examples of FS1 and FS2 were corrected after their first run (see the note above Section 9.6), so the runs with the corrected examples are recorded as version 2.

In [ ]:
# Notes on the worked examples (used by the prompts with "notes": True). A note explains the hard decisions of one example; it is placed in the user message that follows the example, before the next PDF, so the assistant message stays the bare answer.
EXAMPLE_NOTES = [
    "Notes on the example answer above:\n- sex is \"Male\": the report uses \"Mr\" and \"he\".\n- age_at_death is \"43\": digits only.\n- intellectual_disability_flag is null: the report does not address it.\n- mental_health_flag is true: chronic and paranoid schizophrenia are documented.\n- substance_use_flag is true: the report documents abuse of alcohol and multiple drugs.\n- custody_served_flag is true: the deceased was in the lawful custody of Corrective Services NSW at the time of death.\n- domestic_violence_flag is true: the report describes an arrest and charges for domestic violence offences.\n- non_english_speaker_flag and use_of_interpreters_flag are null: the report does not address them.\n- country_of_birth is \"Australia\": the report gives a place of birth in Australia (Dubbo), which is enough without the country being named.\n- preventable_death_flag is null: the coroner does not state whether the death was preventable.\n- place_of_residence and profession are \"\": the report does not state them.",
    "Notes on the example answer above:\n- sex is \"Male\": the report uses \"Mr\" and \"he\".\n- mental_health_flag is true: depression, bipolar disorder and delusional disorder are documented.\n- substance_use_flag is true: the report describes excessive drinking and smoking and treatment for alcohol misuse.\n- custody_served_flag is true: the deceased died in the lawful custody of the NSW Police Force.\n- domestic_violence_flag is null: the alleged offence was a property offence; no domestic violence charge, order or statement is given.\n- preventable_death_flag, non_english_speaker_flag and use_of_interpreters_flag are null: the report does not address them.\n- medications is empty: medication is mentioned only in general terms; no medication is named.\n- country_of_birth is \"\": the report does not state it.\n- profession is \"\": the report describes past work (minister) but does not state an occupation at the time of death.\n- coroners_recommendations_summary is \"\": the coroner made no recommendations.",
]

# Contrastive pairs (a wrong and a right answer) for known errors (used by the prompts with "pairs": True). Invented wording, not taken from any report.
# They are added to the note of the last example, so they appear just before the report to extract.
CONTRASTIVE_PAIRS = "Common mistakes to avoid (invented wording, not taken from any report):\n- The report never mentions interpreters. Wrong: use_of_interpreters_flag = false. Right: null.\n- The report says \"He was 43 years old\". Wrong: age_at_death = \"43 years old\". Right: \"43\".\n- The report refers to \"Mr Smith\" and \"he\" and never states sex directly. Wrong: sex = \"\". Right: \"Male\".\n- The report says the deceased worked as a carpenter until 2015 and was unemployed at the time of death. Wrong: profession = \"carpenter\". Right: \"unemployed\"."


def prompt_examples(prompt: dict) -> list[dict]:
    """Return the worked examples of a few-shot prompt, with notes and pairs if the prompt asks for them.

    Args:
        prompt: An entry of FEW_SHOT_PROMPTS (keys n_shots, and optionally notes and pairs).

    Returns:
        The first `n_shots` examples. With "notes": True each example carries its note under the key `note`; with "pairs": True the wrong-versus-right pairs are appended to the note of the last example.
    """
    examples = [dict(e) for e in FEW_SHOT_EXAMPLES[:prompt["n_shots"]]]
    if prompt.get("notes"):
        for example, note in zip(examples, EXAMPLE_NOTES):
            example["note"] = note
        if prompt.get("pairs"):
            examples[-1]["note"] += "\n\n" + CONTRASTIVE_PAIRS
    return examples


# prompt_id -> the instructions (components), whether the task text is included, the number of worked examples, and whether notes / pairs are added
FEW_SHOT_PROMPTS = {
    "FS1": {"flags": Z4_FLAGS, "include_task_core": True, "n_shots": 1},
    "FS2": {"flags": Z4_FLAGS, "include_task_core": True, "n_shots": 2},
    # prompts with explanatory notes and contrastive pairs
    "FS1_notes": {"flags": Z4_FLAGS, "include_task_core": True, "n_shots": 1, "notes": True},                  # FS1 + a note on the example
    "FS2_notes": {"flags": Z4_FLAGS, "include_task_core": True, "n_shots": 2, "notes": True},                  # FS2 + a note on each example
    "FS2_pairs": {"flags": Z4_FLAGS, "include_task_core": True, "n_shots": 2, "notes": True, "pairs": True},   # FS2_notes + contrastive pairs
    # the same notes and pairs on the Z6 instructions
    "FS2_pairs_Z6": {"flags": Z6_FLAGS, "include_task_core": True, "n_shots": 2, "notes": True, "pairs": True}, # FS2_pairs on the Z6 instructions (variable cards), so that few-shot and zero-shot differ only in the worked examples
}
FEW_SHOT_TO_RUN = [p for p in FEW_SHOT_PROMPTS if p not in SKIP]
if TEST_MODE:        # one prompt per technique is sufficient to test the pipeline and the agreement metrics
    FEW_SHOT_TO_RUN = ["FS2"]
print("few-shot prompts to run:", FEW_SHOT_TO_RUN)


**Preview.** The layout of the messages for FS2, without calling the API:

In [ ]:
_p = FEW_SHOT_PROMPTS["FS2"]
_system, _user = compose(_p["flags"], _p["include_task_core"])
show_messages(build_few_shot_messages(_system, _user, prompt_examples(_p), REPORTS[TEST_REPORT_NO]["file_url"]))
# To see the full texts, call show_messages(..., max_chars=None)


**Preview of FS1_notes, FS2_pairs and FS2_pairs_Z6.** The notes appear in the user messages, before the next PDF.

In [ ]:
for _id in ["FS1_notes", "FS2_pairs", "FS2_pairs_Z6"]:
    _p = FEW_SHOT_PROMPTS[_id]
    _system, _user = compose(_p["flags"], _p["include_task_core"])
    print(f"--- {_id}")
    show_messages(build_few_shot_messages(_system, _user, prompt_examples(_p), REPORTS[TEST_REPORT_NO]["file_url"]))


### 9.5 Smoke test: one few-shot call

One call (prompt FS2, with both worked examples) on the test report. A few-shot call sends three PDF links (two worked examples and the report), so the test confirms that such a request is accepted and shows how many tokens it uses. It is run after the zero-shot smoke test (Section 8.4) and after any change of `MODEL`. The record is saved in the `test_report` folder and is not part of any study dataset.

With `TEST_MODE = False` the cell prints a note and does nothing, so that *Run All* can be used for the study runs.

In [ ]:
if TEST_MODE:      # smoke tests run in test mode only (they use the test report); skipped in study runs
    _p = FEW_SHOT_PROMPTS["FS2"]
    record = run_few_shot_trial(TEST_REPORT_NO, "FS2", _p["flags"], prompt_examples(_p), _p["include_task_core"],
                                dataset="test_report", notes="smoke test")
    print("model:", record["model"], "| worked examples:", _p["n_shots"], "| file number matched:", record["file_number_match"])
    print(f"tokens: {record['usage_input_tokens']} in ({record['usage_cached_tokens']} cached), {record['usage_output_tokens']} out, {record['usage_total_tokens']} total | {record['latency_seconds']} s")
else:
    print("Smoke test skipped: TEST_MODE is False (the smoke tests belong to test mode).")

### Note on the worked examples

After the first prompt development run, the annotations of the two worked examples (`EX1`, file 2021/196736, and `EX2`, file 2023/74097) were checked field by field against the report PDFs. Several values did not match the reports, and the annotations were corrected in the sheet `few_shot_examples` of the reference standard workbook and in `FEW_SHOT_EXAMPLES` (Section 9.1):

- `EX1` `manner_of_death`: the coroner's wording with the name and pronouns removed ("Death occurred after consuming methadone that was supplied by another inmate").
- `EX1` `known_health_issues`: myocardial infarction and cirrhosis of liver added (both named in the report).
- `EX1` `coroners_recommendations_summary`: the numbering "2." of the second recommendation restored.
- `EX2` `profession`: left empty (the report does not state an occupation at the time of death).

Because the worked examples are part of the prompt, the first versions of FS1 and FS2 (with the uncorrected examples) were superseded, and both prompts were run again with the corrected examples. The change was recorded automatically as a new prompt version (Section 6). The trials of the first run remain in the audit log and are not scored (dissertation, Appendix F.3).

### 9.6 Run the few-shot prompts

In test mode only FS2 is run (`FEW_SHOT_TO_RUN`, set in Section 9.4; that cell must be run again after `TEST_MODE` is changed). The number of repeats comes from Section 3 (5 for the test report).

In [ ]:
RUN_NOTE = "External validation: Z7_v2 (best prompt on internal validation), run unchanged"        # optional note, for example "FS2: instructions changed to Z3"

for report_no in TARGET_REPORTS:
    for prompt_id in FEW_SHOT_TO_RUN:
        prompt = FEW_SHOT_PROMPTS[prompt_id]
        for repeat_index in range(1, N_REPEATS + 1):
            record = run_few_shot_trial(report_no, prompt_id, prompt["flags"], prompt_examples(prompt),
                                        prompt["include_task_core"], dataset=DATASET, repeat_index=repeat_index, notes=RUN_NOTE)
            print_trial(record)

## 10. Chain-of-thought

### 10.1 The messages

The layout is the same as for zero-shot, but the user message also holds the request to reason (`COT_INSTRUCTION`, or the request set for the prompt). The model first writes its reasoning in the `reasoning` field and then fills the 28 variables in `extraction` (the `ExtractionWithReasoning` schema, Section 2). The reasoning is additional output, so chain-of-thought trials use more output tokens.

In [ ]:
def build_cot_messages(system_text: str, user_text: str, file_url: str) -> list[dict]:
    """Build the message list of a chain-of-thought prompt.

    Layout:
        system:  the instructions (left out when empty)
        user:    the task text + the request to reason step by step (already part of `user_text`) + the REPORT, as a PDF link

    Args:
        system_text: The system message.
        user_text: The task text, including the chain-of-thought instruction.
        file_url: The link to the report PDF.

    Returns:
        The message list.
    """
    messages = []
    if system_text:
        messages.append({"role": "system", "content": system_text})

    user_content = []
    if user_text:
        user_content.append({"type": "input_text", "text": user_text})
    user_content.append({"type": "input_file", "file_url": file_url})
    messages.append({"role": "user", "content": user_content})
    return messages

### 10.2 The chain-of-thought trial function

The reasoning is stored in `reasoning` and the 28 variables in `output`, so `output` has the same shape for every technique.

In [ ]:
def run_cot_trial(report_no: int, prompt_id: str, flags: dict, include_task_core: bool = True, cot_instruction: str = COT_INSTRUCTION, *,
                  dataset: str, repeat_index: int = 1, notes: str = "", response_schema=ExtractionWithReasoning) -> dict:
    """Run one chain-of-thought call on one report and save it.

    Args:
        report_no: The report number.
        prompt_id: The prompt name.
        flags: The prompt components to switch on.
        include_task_core: Whether to include the task text.
        cot_instruction: The request to reason step by step, added to the user text.
        dataset: The set the trial belongs to.
        repeat_index: The repeat number of this prompt on this report.
        notes: Free-text note stored with the trial.
        response_schema: The shape of the answer: ExtractionWithReasoning (a `reasoning` text, then the extraction) or ExtractionWithEvidence (an `evidence` list, then the extraction).

    Returns:
        The saved record, as a dict.
    """
    report = check_trial_allowed(report_no, dataset)                                         # 1. check
    system_text, user_text = compose(flags, include_task_core)
    user_text = (user_text + "\n\n" + cot_instruction).strip()                               # the request to reason step by step
    messages = build_cot_messages(system_text, user_text, report["file_url"])                # 2. messages
    response, seconds = call_model(messages, response_schema)                        # 3. call
    return log_trial(response, seconds, report_no=report_no, dataset=dataset, technique="chain_of_thought",   # 4. log
                     prompt_id=prompt_id, repeat_index=repeat_index, flags=flags, include_task_core=include_task_core,
                     system_text=system_text, user_text=user_text, use_cot=True, notes=notes,
                     schema_name=None if response_schema is ExtractionWithReasoning else response_schema.__name__)

### 10.3 The chain-of-thought prompts

Each prompt has its own instructions (`flags`) and its own request to reason (`cot_instruction`), so that the wording of the request can vary (dissertation, Table 6 and Appendix G, Table G3). A change to either is recorded automatically as a new version of that prompt (Section 6).

| Prompt | Instructions | Request to reason | Form of the answer |
|---|---|---|---|
| **COT1** | Z4 | reason step by step | `reasoning` text, then `extraction` |
| **COT1_evidence** | Z4 | for each flag, sex, age at death and profession: quote the relevant sentence and state the rule that applies, before the value | `reasoning` text, then `extraction` |
| **COT1_struct** | Z4 | the same, as a structured list | `evidence` list (field, quotation, rule applied), then `extraction` |
| **COT2_A** | Z6 (decision cards) | for all 28 variables: the report's exact words, the card item that applies, and the value | `reasoning` text, then `extraction` |
| **COT2_B** | Z6 | as COT2_A, for the 14 variables that require judgement | `reasoning` text, then `extraction` |
| **COT2_C** | Z6 | as COT2_A, as a structured list | `evidence` list, then `extraction` |

A prompt with the entry `"schema": ExtractionWithEvidence` changes the form of the answer; the extraction is read and scored in the same way. The evidence list is saved as JSON text in the field `reasoning` of the trial record.

In [ ]:
# Requests to reason (added to the user message after the task text).
COT_EVIDENCE_INSTRUCTION = "Before giving the extraction, use the `reasoning` field as follows. For each boolean flag, quote the sentence in the report that bears on it (or write \"none\"), then state which rule applies: explicitly confirmed means true, explicitly denied means false, not addressed means null. Do the same for sex, age_at_death and profession. Do not add commentary elsewhere. Then give the completed extraction in the `extraction` field."

COT_STRUCT_INSTRUCTION = "Complete the `evidence` field first. It is a list with one entry for each boolean flag, for sex, for age_at_death and for profession. Each entry has three parts: `field` (the field name), `quote` (the sentence from the report that bears on the field, or \"none\") and `rule_applied` (explicitly confirmed means true, explicitly denied means false, not addressed means null). Then give the completed extraction in the `extraction` field."

COT2_A_INSTRUCTION = "Before giving the extraction, use the `reasoning` field to work through every variable in the order listed. Write one line for each variable in this form: variable name | evidence: the exact words from the report that bear on it, or \"none\" | card check: the \"Counts\" or \"Does not count\" item that applies, or that the report is silent | value. Quote the report exactly; do not paraphrase the evidence. For list and text variables, quote the passage the value is taken from. Then give the completed extraction in the `extraction` field, using exactly the values worked out in the `reasoning` field."

COT2_B_INSTRUCTION = "Before giving the extraction, use the `reasoning` field for the variables that need a judgement: the eight boolean flags, sex, age_at_death, country_of_birth, place_of_residence, profession and level_of_education. Write one line for each in this form: variable name | evidence: the exact words from the report that bear on it, or \"none\" | card check: the \"Counts\" or \"Does not count\" item that applies, or that the report is silent | value. Quote the report exactly; do not paraphrase the evidence. The other variables need no reasoning: take them directly from the report using their cards. Then give the completed extraction in the `extraction` field, using exactly the values worked out in the `reasoning` field."

COT2_C_INSTRUCTION = "Complete the `evidence` field first. It is a list with one entry for every variable, in the order listed. Each entry has three parts: `field` (the variable name), `quote` (the exact words from the report that bear on the variable, or \"none\") and `rule_applied` (the \"Counts\" or \"Does not count\" item that applies, or that the report is silent). Then give the completed extraction in the `extraction` field, using exactly the values supported by the evidence."

# prompt_id -> the instructions (components), whether the task text is included, the request to reason, and (optionally) the shape of the answer
COT_PROMPTS = {
    "COT1": {"flags": Z4_FLAGS, "include_task_core": True, "cot_instruction": COT_INSTRUCTION},
    # evidence-based prompts
    "COT1_evidence": {"flags": Z4_FLAGS, "include_task_core": True, "cot_instruction": COT_EVIDENCE_INSTRUCTION},                           # Z4 + evidence-first request in the `reasoning` text
    "COT1_struct": {"flags": Z4_FLAGS, "include_task_core": True, "cot_instruction": COT_STRUCT_INSTRUCTION, "schema": ExtractionWithEvidence},  # Z4 + structured `evidence` list
    "COT2_A": {"flags": Z6_FLAGS, "include_task_core": True, "cot_instruction": COT2_A_INSTRUCTION},                                       # cards (Z6) + evidence, card check and value for all 28 variables
    "COT2_B": {"flags": Z6_FLAGS, "include_task_core": True, "cot_instruction": COT2_B_INSTRUCTION},                                       # cards (Z6) + the same for 14 judgement variables only
    "COT2_C": {"flags": Z6_FLAGS, "include_task_core": True, "cot_instruction": COT2_C_INSTRUCTION, "schema": ExtractionWithEvidence},        # cards (Z6) + structured `evidence` list
}
COT_TO_RUN = [p for p in COT_PROMPTS if p not in SKIP]
if TEST_MODE:        # one prompt per technique is sufficient to test the pipeline and the agreement metrics
    COT_TO_RUN = ["COT1"]
print("chain-of-thought prompts to run:", COT_TO_RUN)


**Preview.** The layout of the messages for COT1, without calling the API:

In [ ]:
_p = COT_PROMPTS["COT1"]
_system, _user = compose(_p["flags"], _p["include_task_core"])
show_messages(build_cot_messages(_system, (_user + "\n\n" + _p["cot_instruction"]).strip(), REPORTS[TEST_REPORT_NO]["file_url"]))
# To see the full texts, call show_messages(..., max_chars=None)


**Preview of COT1_evidence and COT2_A.**

In [ ]:
for _id in ["COT1_evidence", "COT2_A"]:
    _p = COT_PROMPTS[_id]
    _system, _user = compose(_p["flags"], _p["include_task_core"])
    print(f"--- {_id}")
    show_messages(build_cot_messages(_system, (_user + "\n\n" + _p["cot_instruction"]).strip(), REPORTS[TEST_REPORT_NO]["file_url"]))


### 10.4 Smoke test: one chain-of-thought call

One call (prompt COT1) on the test report. In addition to the model name, the PDF link and the token counts, the test confirms that the reasoning is returned together with the 28 variables. It is run after the zero-shot smoke test (Section 8.4) and after any change of `MODEL`. The record is saved in the `test_report` folder and is not part of any study dataset.

With `TEST_MODE = False` the cell prints a note and does nothing, so that *Run All* can be used for the study runs.

In [ ]:
if TEST_MODE:      # smoke tests run in test mode only (they use the test report); skipped in study runs
    _p = COT_PROMPTS["COT1"]
    record = run_cot_trial(TEST_REPORT_NO, "COT1", _p["flags"], _p["include_task_core"], _p["cot_instruction"],
                           dataset="test_report", notes="smoke test")
    print("model:", record["model"], "| file number matched:", record["file_number_match"], "| reasoning returned:", bool(record.get("reasoning")))
    print(f"tokens: {record['usage_input_tokens']} in ({record['usage_cached_tokens']} cached), {record['usage_output_tokens']} out ({record['usage_reasoning_tokens']} reasoning), {record['usage_total_tokens']} total | {record['latency_seconds']} s")
else:
    print("Smoke test skipped: TEST_MODE is False (the smoke tests belong to test mode).")

### 10.5 Run the chain-of-thought prompts

In test mode only COT1 is run (`COT_TO_RUN`, set in Section 10.3; that cell must be run again after `TEST_MODE` is changed). The number of repeats comes from Section 3 (5 for the test report).

In [ ]:
RUN_NOTE = "External validation: Z7_v2 (best prompt on internal validation), run unchanged"        # optional note, for example "COT1: reasoning request reworded"

for report_no in TARGET_REPORTS:
    for prompt_id in COT_TO_RUN:
        prompt = COT_PROMPTS[prompt_id]
        for repeat_index in range(1, N_REPEATS + 1):
            record = run_cot_trial(report_no, prompt_id, prompt["flags"], prompt["include_task_core"], prompt["cot_instruction"],
                                   dataset=DATASET, repeat_index=repeat_index, notes=RUN_NOTE,
                                   response_schema=prompt.get("schema", ExtractionWithReasoning))
            print_trial(record)

## 11. Export: prompt tracker, review sheet, token usage and prompt versions

Reads every trial logged for `DATASET` and writes two spreadsheets to its results folder: `prompt_tracker.xlsx` (one row per trial: the prompt, its version and settings, and its token usage) and `human_review.xlsx` (one row per trial: the prompt version, the report, the full text of the prompt sent, the repeat, and one column per extracted variable, for reading the outputs). The cell can be run again after further trials. The last lines summarise token usage per prompt version, followed by the list of all recorded prompt versions.

In [ ]:
log_df = pd.read_json(dataset_dir(DATASET) / "audit_log.jsonl", lines=True)
for _col, _default in [("prompt_version", 0), ("prompt_hash", ""), ("reasoning_effort", None), ("temperature", None), ("top_p", None),
                       ("max_output_tokens", None), ("verbosity", None), ("service_tier", None), ("response_status", None)]:                  # records written before these fields were logged
    if _col not in log_df:
        log_df[_col] = _default

factor_cols = pd.json_normalize(log_df["factors"]).add_prefix("factor__")
tracker_df = pd.concat(
    [log_df[["trial_id", "dataset", "report_no", "technique", "n_shots", "prompt_id", "prompt_version", "prompt_hash", "repeat_index", "model", "reasoning_effort", "temperature", "top_p", "max_output_tokens", "verbosity", "service_tier", "notes",
             "usage_input_tokens", "usage_cached_tokens", "usage_output_tokens", "usage_reasoning_tokens",
             "usage_total_tokens", "latency_seconds", "file_number_match", "response_status"]], factor_cols], axis=1)
tracker_df.to_excel(dataset_dir(DATASET) / "prompt_tracker.xlsx", index=False)


def full_prompt_text(row) -> str:
    """Return the full text of the prompt that was sent in one trial, as one string.

    The string holds the system message and the user message. For a few-shot trial, a final line states how many worked examples were sent (the examples are re-sent with every call) and the links of their reports. The text is cut at 32,000 characters, the limit of one Excel cell.

    Args:
        row: One record of the audit log.

    Returns:
        The prompt text.
    """
    system = row["system_text"] if "system_text" in row and isinstance(row["system_text"], str) else ""
    user = row["user_text"] if isinstance(row["user_text"], str) else ""
    text = (f"SYSTEM MESSAGE:\n{system}\n\n" if system else "NO SYSTEM MESSAGE\n\n") + f"USER MESSAGE:\n{user}"
    if row["n_shots"] and row["n_shots"] > 0:
        urls = row["example_urls"] if isinstance(row["example_urls"], list) else []
        text += f"\n\n[{int(row['n_shots'])} worked example(s) sent before the report: " + "; ".join(map(str, urls)) + "]"
    return text[:32000]


output_df = pd.json_normalize(log_df["output"]).apply(lambda col: col.map(lambda v: "; ".join(map(str, v)) if isinstance(v, list) else v))
review_df = pd.concat(
    [log_df[["prompt_id", "prompt_version", "report_no"]],
     log_df.apply(full_prompt_text, axis=1).rename("prompt_text"),                                  # the full prompt, between report_no and repeat_index
     log_df[["repeat_index", "trial_id", "technique", "n_shots", "notes"]], output_df], axis=1
).sort_values(["prompt_id", "prompt_version", "report_no", "repeat_index", "trial_id"])
with pd.ExcelWriter(dataset_dir(DATASET) / "human_review.xlsx", engine="openpyxl") as _xw:
    review_df.to_excel(_xw, sheet_name="human_review", index=False)
    _ws = _xw.sheets["human_review"]
    _ws.freeze_panes = "A2"
    _ws.column_dimensions["D"].width = 60                                                           # the prompt text sits in column D

# average tokens per trial for each prompt version
_keys = ["technique", "prompt_id", "prompt_version"]
token_summary = (log_df.groupby(_keys)[["usage_input_tokens", "usage_cached_tokens", "usage_output_tokens", "usage_total_tokens"]]
                 .mean().round(0).astype(int).assign(trials=log_df.groupby(_keys).size()))
token_summary

In [ ]:
show_prompt_versions()

## 12. Steps after prompt development

1. The prompt development results were scored with `evaluation_pipeline_final.ipynb`, which ranks the prompt versions overall and within each technique (a version is shown as, for example, `Z4_v2`) and reports the agreement between repeats.
2. The best eligible version of each technique was selected with the rule fixed in advance: the highest unweighted mean of the 28 variable scores, with ties going to the shorter prompt and then to the prompt with fewer worked examples. The three candidates were `Z7_v2`, `FS2_pairs_Z6_v2` and `COT2_A_v2`.
3. The wording in the notebook was checked against the selected versions. `show_prompt_version("Z4", 1)` prints the full text of any earlier version, so that it can be restored.
4. Internal validation: `"internal_validation"` was added to `ALLOWED_DATASETS`, `DATASET` was set to `"internal_validation"` (5 repeats), and only the three candidates were run. Each trial prints its prompt version, and a warning appears if a wording was not run in prompt development. The results were scored and the single best prompt, `Z7_v2`, was selected.
5. External validation: `"external_validation"` is in `ALLOWED_DATASETS`, `DATASET = "external_validation"` (1 repeat), and `NOT_IN_EXTERNAL_VALIDATION` holds `FS2_pairs_Z6` and `COT2_A`, so that *Run All* runs only `Z7_v2` on the 40 external validation reports (40 calls). The results were scored with `evaluation_pipeline_final.ipynb`.

OpenAI applies prompt caching automatically to repeated prompt beginnings (`usage_cached_tokens` shows how much of the input was served from the cache). Caching affects cost and speed, but not the extracted values.